# Sudoku-04 : Recuit Simule (Rust)

**Niveau** : Métaheuristique | **Durée** : ~20 min | **Prerequis** : Sudoku-00 Environment

## Navigation

| << Précédent | [Index](README.md) | Suivant >> |
|-------------|---------------------|-----------|
| [Sudoku-03-Genetic-Python](Sudoku-03-Genetic-Python.ipynb) | | [Sudoku-05-PSO-Python](Sudoku-05-PSO-Python.ipynb) |

## Objectifs d'apprentissage

A la fin de ce notebook, vous saurez :
1. Formuler la resolution de Sudoku comme un problème d'optimisation
2. Définir une fonction d'energie comptant les violations de contraintes
3. Construire un voisinage par echange de cellules
4. Implementer un solveur par recuit simule avec la bibliotheque `simanneal`
5. Analyser les forces et limites du recuit simule pour le Sudoku

***

Ce notebook implemente un solveur de Sudoku utilisant le recuit simule (Simulated Annealing) en Rust.
C'est l'équivalent Rust du notebook C# `Sudoku-04-SimulatedAnnealing-CSharp.ipynb`.

## Introduction

Le **recuit simule** (Simulated Annealing, SA) est une métaheuristique inspiree du processus metallurgique de recuit : un metal est chauffe puis refroidi lentement pour atteindre un état cristallin optimal.

### Principe

- **État** : Une grille entierement remplie (potentiellement avec des erreurs)
- **Energie** : Mesure le nombre de violations de contraintes
- **Temperature** : Contrôle l'acceptation de mouvements degradants
- **Refroidissement** : Réduit progressivement la temperature

### Critere d'acceptation de Metropolis

$$P(\text{accepter}) = \begin{cases} 1 & \text{si } \Delta E \leq 0 \\ e^{-\Delta E / T} & \text{si } \Delta E > 0 \end{cases}$$

ou $\Delta E = E(\text{voisin}) - E(\text{courant})$ et $T$ est la temperature courante.

## Installation

Aucune crate externe requise : stdlib Rust uniquement.
La boucle `anneal()` de `simanneal` est réécrite en std (section 5).

In [2]:
// Imports
// Note: La bibliotheque simanneal est optionnelle
// Ce notebook utilise principalement l'implementation manuelle
// Sous Rust (evcxr), la stdlib est implicite : aucun use externe.
// La boucle anneal() de simanneal est reecrite en std (section 5).
println!("simanneal importe avec succes");

simanneal importe avec succes


Configuration du chemin vers les fichiers de puzzles.

In [3]:
// Configuration du chemin vers les puzzles
// Resolution robuste du repertoire du notebook (CWD peut differer sous Papermill)
use std::path::PathBuf;

let here = std::env::current_dir().unwrap();
let mut candidates = vec![
    here.clone(),
    here.join("MyIA.AI.Notebooks").join("Sudoku"),
];
if let Some(parent) = here.parent() {
    candidates.push(parent.to_path_buf().join("Sudoku"));
}

let mut puzzles_dir = PathBuf::from("Puzzles");
for candidate in &candidates {
    if candidate.join("Puzzles").is_dir() {
        puzzles_dir = candidate.join("Puzzles");
        break;
    }
}

if puzzles_dir.is_dir() {
    println!("Dossier Puzzles: {}", puzzles_dir.file_name().unwrap_or_default().to_string_lossy());
} else {
    println!("ATTENTION: Dossier Puzzles non trouve");
};

Dossier Puzzles: Puzzles


## 1. Classe SudokuGrid

Représentation d'une grille de Sudoku avec méthodes pour calculer l'energie et generer des voisins.

In [4]:
// Grid + from_string + show + load_puzzles
#[derive(Clone)]
struct Grid { cells: [[u8; 9]; 9] }

impl Grid {
    fn from_string(s: &str) -> Grid {
        let s = s.replace('.', "0").replace(' ', "").replace('\n', "");
        assert!(s.len() == 81, "La chaine doit avoir 81 caracteres");
        let mut cells = [[0u8; 9]; 9];
        for (i, ch) in s.chars().enumerate() {
            cells[i / 9][i % 9] = ch.to_digit(10).unwrap() as u8;
        }
        Grid { cells }
    }

    fn to_string(&self) -> String {
        let mut s = String::new();
        for r in 0..9 {
            for c in 0..9 {
                s.push((b'0' + self.cells[r][c]) as char);
            }
        }
        s
    }

    fn show(&self) -> String {
        let mut lines = Vec::new();
        for r in 0..9 {
            if r > 0 && r % 3 == 0 {
                lines.push("-".repeat(21));
            }
            let mut row_str = String::new();
            for c in 0..9 {
                if c > 0 && c % 3 == 0 {
                    row_str.push_str("| ");
                }
                let val = self.cells[r][c];
                if val == 0 {
                    row_str.push('.');
                } else {
                    row_str.push((b'0' + val) as char);
                }
                row_str.push(' ');
            }
            lines.push(row_str);
        }
        lines.join("\n")
    }
}

fn load_puzzles(filepath: &str, max: Option<usize>) -> Vec<String> {
    let content = std::fs::read_to_string(filepath).unwrap();
    let mut puzzles = Vec::new();
    for line in content.lines() {
        let line = line.trim();
        if line.len() >= 81 {
            puzzles.push(line[..81].to_string());
            if let Some(m) = max {
                if puzzles.len() >= m {
                    break;
                }
            }
        }
    }
    puzzles
}

// Charger les puzzles
let here = std::env::current_dir().unwrap();
let mut candidates = vec![
    here.clone(),
    here.join("MyIA.AI.Notebooks").join("Sudoku"),
];
if let Some(parent) = here.parent() {
    candidates.push(parent.to_path_buf().join("Sudoku"));
}
let mut puzzles_dir = std::path::PathBuf::from("Puzzles");
for candidate in &candidates {
    if candidate.join("Puzzles").is_dir() {
        puzzles_dir = candidate.join("Puzzles");
        break;
    }
}

let easy_puzzles = load_puzzles(&puzzles_dir.join("Sudoku_Easy51.txt").to_string_lossy(), Some(5));
println!("Puzzles charges: {}", easy_puzzles.len());

let test_grid = Grid::from_string(&easy_puzzles[0]);
println!("\nGrille de test:");
println!("{}", test_grid.show());

Puzzles charges: 5


Grille de test:


9 . 2 | . . 5 | 4 . 3 


1 . . | . 6 3 | . 2 5 


5 . 8 | 4 . 7 | . 6 . 


---------------------


... (3 outputs left out)

---------------------


2 4 . | 5 3 . | 6 . . 


7 . 5 | 2 . . | 3 . 4 


. 8 . | . 4 1 | 9 5 . 


### Lecture du corpus : 5 puzzles, la grille témoin en tête

**5 puzzles chargés**, et la grille de test affichée est la **grille témoin de la série** — comptez : 9,2,5,4,3 en première ligne, **45 indices** au total, le même puzzle que Sudoku-06 (AIMA/MAC) et Sudoku-11 (Choco). L'uniformité du témoin est précieuse : quand ce notebook mesurera son solveur, la comparaison avec les autres approches de la série portera sur des entrées identiques. Le point `.` marque une cellule libre — 36 déductions à faire ici, mais pour le recuit simulé ce ne sont pas des déductions : c'est une **configuration initiale à perturber**.

## 2. Fonction d'Energie

L'energie compte le nombre de violations de contraintes dans les colonnes et les blocs 3x3. Les lignes sont garanties valides par construction (permutations).

## Exercice : Calculer l'energie d'une grille partiellement remplie

**Objectif**
Implementez une variante de `compute_energy` qui calcule le nombre de conflits
uniquement pour les lignes, colonnes et blocs non complets.

**Indice**
Parcourir chaque ligne/colonne/bloc et compter les doublons parmi les valeurs non nulles.

In [5]:
// EXERCICE : Calculer l'energie d'une grille partiellement remplie
fn compute_partial_energy(_grid: &[[u8; 9]; 9]) -> i32 {
    // TODO: Compter les conflits (doublons) uniquement dans les
    // lignes/colonnes/blocs qui contiennent des valeurs non nulles
    0 // TODO etudiant
}
println!("Exercice a completer");

Exercice a completer


In [6]:
// compute_energy : doublons colonnes + blocs
#[derive(Clone)]
struct Grid { cells: [[u8; 9]; 9] }

impl Grid {
    fn from_string(s: &str) -> Grid {
        let s = s.replace('.', "0").replace(' ', "").replace('\n', "");
        assert!(s.len() == 81, "La chaine doit avoir 81 caracteres");
        let mut cells = [[0u8; 9]; 9];
        for (i, ch) in s.chars().enumerate() {
            cells[i / 9][i % 9] = ch.to_digit(10).unwrap() as u8;
        }
        Grid { cells }
    }
}

fn compute_energy(grid: &[[u8; 9]; 9]) -> i32 {
    let mut energy = 0i32;

    // Doublons dans les colonnes
    for col in 0..9 {
        let mut counts = [0i32; 10];
        for r in 0..9 {
            let v = grid[r][col];
            if v > 0 { counts[v as usize] += 1; }
        }
        for v in 1..10 {
            energy += (counts[v] - 1).max(0); // doublons uniquement (clamp: valeur absente = 0)
        }
    }

    // Doublons dans les blocs 3x3
    for box_row in 0..3 {
        for box_col in 0..3 {
            let mut counts = [0i32; 10];
            for r in 0..3 {
                for c in 0..3 {
                    let v = grid[box_row * 3 + r][box_col * 3 + c];
                    if v > 0 { counts[v as usize] += 1; }
                }
            }
            for v in 1..10 {
                energy += (counts[v] - 1).max(0); // doublons uniquement (clamp: valeur absente = 0)
            }
        }
    }

    energy
}

// Charger le puzzle de test
let here = std::env::current_dir().unwrap();
let mut candidates = vec![
    here.clone(),
    here.join("MyIA.AI.Notebooks").join("Sudoku"),
];
if let Some(parent) = here.parent() {
    candidates.push(parent.to_path_buf().join("Sudoku"));
}
let mut puzzles_dir = std::path::PathBuf::from("Puzzles");
for candidate in &candidates {
    if candidate.join("Puzzles").is_dir() {
        puzzles_dir = candidate.join("Puzzles");
        break;
    }
}
let content = std::fs::read_to_string(puzzles_dir.join("Sudoku_Easy51.txt")).unwrap();
let first_puzzle = content.lines().find(|l| l.trim().len() >= 81).unwrap().trim()[..81].to_string();

let test_grid = Grid::from_string(&first_puzzle);
println!("Fonction compute_energy definie.");
println!("Energie de la grille de test: {}", compute_energy(&test_grid.cells));

Fonction compute_energy definie.


Energie de la grille de test: 0


### Interpretation : Fonction d'energie

| Aspect | Valeur | Signification |
|--------|--------|---------------|
| Energie = 0 | Solution valide | Aucun doublon colonnes/blocs |
| Energie > 0 | Grille invalide | Nombre total de violations |

L'objectif est de reduire l'energie a 0 par echanges successifs.

> **Fonction d'energie** : on compte, pour chaque colonne et chaque bloc 3x3, le nombre de doublons (`counts[v] - 1` pour les valeurs presentes, `0` pour les valeurs absentes via `.max(0)`). Les lignes sont valides par construction (permutations de 1-9), donc seules les colonnes et les blocs peuvent porter des violations. L'energie de la grille de test (partiellement remplie) vaut **0** ci-dessus car les cellules fixees par le puzzle ne créent pas encore de conflit ; en revanche, une fois la grille completee par permutations (cellule suivante), l'energie devient positive et le recuit doit la minimiser.

## 3. Initialisation de la Grille

Chaque ligne est initialisee comme une permutation de 1-9, en respectant les cellules fixes.

In [7]:
// initialize_grid : permutations par ligne
// SplitMix64 — RNG std simple, aucune crate.
struct Rng { state: u64 }
impl Rng {
    fn new(seed: u64) -> Rng { Rng { state: seed } }
    fn next_u64(&mut self) -> u64 {
        self.state = self.state.wrapping_add(0x9e37_79b9_7f4a_7c15);
        let mut z = self.state;
        z = (z ^ (z >> 30)).wrapping_mul(0xbf58_476d_1ce4_e5b9);
        z = (z ^ (z >> 27)).wrapping_mul(0x94d0_49bb_1331_11eb);
        z ^ (z >> 31)
    }
    fn randbelow(&mut self, n: u64) -> u64 { self.next_u64() % n }
    fn shuffle<T>(&mut self, x: &mut [T]) {
        for i in (1..x.len()).rev() {
            let j = self.randbelow(i as u64 + 1) as usize;
            x.swap(i, j);
        }
    }
}

#[derive(Clone)]
struct Grid { cells: [[u8; 9]; 9] }

impl Grid {
    fn from_string(s: &str) -> Grid {
        let s = s.replace('.', "0").replace(' ', "").replace('\n', "");
        assert!(s.len() == 81, "La chaine doit avoir 81 caracteres");
        let mut cells = [[0u8; 9]; 9];
        for (i, ch) in s.chars().enumerate() {
            cells[i / 9][i % 9] = ch.to_digit(10).unwrap() as u8;
        }
        Grid { cells }
    }
    fn show(&self) -> String {
        let mut lines = Vec::new();
        for r in 0..9 {
            if r > 0 && r % 3 == 0 {
                lines.push("-".repeat(21));
            }
            let mut row_str = String::new();
            for c in 0..9 {
                if c > 0 && c % 3 == 0 {
                    row_str.push_str("| ");
                }
                let val = self.cells[r][c];
                if val == 0 {
                    row_str.push('.');
                } else {
                    row_str.push((b'0' + val) as char);
                }
                row_str.push(' ');
            }
            lines.push(row_str);
        }
        lines.join("\n")
    }
}

fn compute_energy(grid: &[[u8; 9]; 9]) -> i32 {
    let mut energy = 0i32;
    for col in 0..9 {
        let mut counts = [0i32; 10];
        for r in 0..9 {
            let v = grid[r][col];
            if v > 0 { counts[v as usize] += 1; }
        }
        for v in 1..10 { energy += (counts[v] - 1).max(0); }
    }
    for br in 0..3 {
        for bc in 0..3 {
            let mut counts = [0i32; 10];
            for r in 0..3 {
                for c in 0..3 {
                    let v = grid[br * 3 + r][bc * 3 + c];
                    if v > 0 { counts[v as usize] += 1; }
                }
            }
            for v in 1..10 { energy += (counts[v] - 1).max(0); }
        }
    }
    energy
}

fn initialize_grid(puzzle: &Grid, rng: &mut Rng) -> (Grid, [[bool; 9]; 9]) {
    let mut grid = puzzle.clone();
    let mut is_fixed = [[false; 9]; 9];
    for r in 0..9 {
        for c in 0..9 {
            is_fixed[r][c] = puzzle.cells[r][c] > 0;
        }
    }
    for row in 0..9 {
        // Identifier les valeurs fixes et manquantes
        let mut fixed = [false; 10];
        for c in 0..9 {
            let v = grid.cells[row][c];
            if v > 0 { fixed[v as usize] = true; }
        }
        let mut missing: Vec<u8> = (1..=9).filter(|&v| !fixed[v as usize]).collect();

        // Melanger les valeurs manquantes
        rng.shuffle(&mut missing);

        // Remplir les cellules vides
        let mut i = 0;
        for c in 0..9 {
            if grid.cells[row][c] == 0 {
                grid.cells[row][c] = missing[i];
                i += 1;
            }
        }
    }
    (grid, is_fixed)
}

// Démo sur le puzzle 1
let here = std::env::current_dir().unwrap();
let mut candidates = vec![
    here.clone(),
    here.join("MyIA.AI.Notebooks").join("Sudoku"),
];
if let Some(parent) = here.parent() {
    candidates.push(parent.to_path_buf().join("Sudoku"));
}
let mut puzzles_dir = std::path::PathBuf::from("Puzzles");
for candidate in &candidates {
    if candidate.join("Puzzles").is_dir() {
        puzzles_dir = candidate.join("Puzzles");
        break;
    }
}
let content = std::fs::read_to_string(puzzles_dir.join("Sudoku_Easy51.txt")).unwrap();
let first_puzzle = content.lines().find(|l| l.trim().len() >= 81).unwrap().trim()[..81].to_string();

let test_grid = Grid::from_string(&first_puzzle);
let mut rng = Rng::new(42);
let (init_grid, _is_fixed) = initialize_grid(&test_grid, &mut rng);

println!("Grille initialisee (chaque ligne est une permutation de 1-9):");
println!("{}", init_grid.show());
println!("\nEnergie initiale: {}", compute_energy(&init_grid.cells));
println!("Objectif: energie = 0");

Grille initialisee (chaque ligne est une permutation de 1-9):


9 7 2 | 1 8 5 | 4 6 3 


1 8 9 | 7 6 3 | 4 2 5 


5 1 8 | 4 9 7 | 3 6 2 


---------------------


5 2 6 | 3 4 9 | 8 7 1 


... (4 outputs left out)

7 8 5 | 2 9 1 | 3 6 4 


6 8 3 | 7 4 1 | 9 5 2 


Energie initiale: 35


Objectif: energie = 0


### Lecture de l'initialisation : des lignes parfaites pour concentrer le défaut

La grille initiale est construite en **complétant chaque ligne par une permutation de 1-9** (les indices fixés en place, les trous remplis). Conséquence immédiate : chaque ligne contient 1-9 exactement une fois — **l'énergie des lignes est nulle par construction**, et tout le conflit se concentre dans les colonnes et les blocs. D'où l'énergie initiale : comptez la première colonne de la sortie — les doublons y sont visibles à l'œil. Cette initialisation n'est pas neutre : en réduisant l'espace de recherche aux permutations de lignes (au lieu de toutes les grilles), elle rend le voisinage « échange dans une ligne » **fermé** — un échange ne peut jamais casser une ligne, seulement en déplacer les conflits.


Notez enfin le choix pédagogique de l'énergie initiale : elle n'est pas nulle, mais elle est **petite et lisible** — on peut la décomposer à la main (quelques conflits dans la première colonne, le reste en colonnes et blocs). C'est cette lisibilité qui permet de suivre l'algorithme pas à pas dans les sections suivantes, là où une initialisation aléatoire produirait une énergie de l'ordre de 150 impossible à décortiquer.

## Exercice : Generer un voisin par echange de bloc

**Objectif**
Implementez un opérateur de voisinage qui echange deux valeurs non fixees
dans le même bloc 3x3 (au lieu de la même ligne).

**Indice**
Choisissez un bloc aléatoire, puis deux cellules non fixees dans ce bloc,
et echangez leurs valeurs.

In [8]:
// EXERCICE : Generer un voisin par echange de bloc
#[derive(Clone)]
struct Grid { cells: [[u8; 9]; 9] }
struct Rng { state: u64 }
impl Rng {
    fn new(seed: u64) -> Rng { Rng { state: seed } }
    fn next_u64(&mut self) -> u64 {
        self.state = self.state.wrapping_add(0x9e37_79b9_7f4a_7c15);
        let mut z = self.state;
        z = (z ^ (z >> 30)).wrapping_mul(0xbf58_476d_1ce4_e5b9);
        z = (z ^ (z >> 27)).wrapping_mul(0x94d0_49bb_1331_11eb);
        z ^ (z >> 31)
    }
    fn randbelow(&mut self, n: u64) -> u64 { self.next_u64() % n }
    fn shuffle<T>(&mut self, x: &mut [T]) {
        for i in (1..x.len()).rev() {
            let j = self.randbelow(i as u64 + 1) as usize;
            x.swap(i, j);
        }
    }
}

fn generate_block_neighbor(_grid: &Grid, _is_fixed: &[[bool; 9]; 9], _rng: &mut Rng) -> Option<(Grid, (usize, usize))> {
    // TODO: Echangez deux valeurs non fixees dans un meme bloc 3x3
    // Retournez (nouvelle_grille, position_echangee)
    None // TODO etudiant
}
println!("Exercice a completer");

Exercice a completer


## 4. Voisinage par Echange

Le voisinage est défini par l'echange de deux cellules non fixes dans une même ligne. Cela preserve la propriete de permutation de la ligne.

In [9]:
// generate_neighbor : echange intra-ligne
// SplitMix64 — RNG std simple, aucune crate.
struct Rng { state: u64 }
impl Rng {
    fn new(seed: u64) -> Rng { Rng { state: seed } }
    fn next_u64(&mut self) -> u64 {
        self.state = self.state.wrapping_add(0x9e37_79b9_7f4a_7c15);
        let mut z = self.state;
        z = (z ^ (z >> 30)).wrapping_mul(0xbf58_476d_1ce4_e5b9);
        z = (z ^ (z >> 27)).wrapping_mul(0x94d0_49bb_1331_11eb);
        z ^ (z >> 31)
    }
    fn randbelow(&mut self, n: u64) -> u64 { self.next_u64() % n }
    fn random(&mut self) -> f64 { (self.next_u64() >> 11) as f64 / (1u64 << 53) as f64 }
    fn shuffle<T>(&mut self, x: &mut [T]) {
        for i in (1..x.len()).rev() {
            let j = self.randbelow(i as u64 + 1) as usize;
            x.swap(i, j);
        }
    }
    fn sample2(&mut self, population: &[usize]) -> (usize, usize) {
        let n = population.len();
        let mut pool: Vec<usize> = population.to_vec();
        let mut result = [0usize; 2];
        for i in 0..2 {
            let j = self.randbelow((n - i) as u64) as usize;
            result[i] = pool[j];
            pool[j] = pool[n - i - 1];
        }
        (result[0], result[1])
    }
}

#[derive(Clone)]
struct Grid { cells: [[u8; 9]; 9] }

impl Grid {
    fn from_string(s: &str) -> Grid {
        let s = s.replace('.', "0").replace(' ', "").replace('\n', "");
        assert!(s.len() == 81, "La chaine doit avoir 81 caracteres");
        let mut cells = [[0u8; 9]; 9];
        for (i, ch) in s.chars().enumerate() {
            cells[i / 9][i % 9] = ch.to_digit(10).unwrap() as u8;
        }
        Grid { cells }
    }
    fn show(&self) -> String {
        let mut lines = Vec::new();
        for r in 0..9 {
            if r > 0 && r % 3 == 0 {
                lines.push("-".repeat(21));
            }
            let mut row_str = String::new();
            for c in 0..9 {
                if c > 0 && c % 3 == 0 {
                    row_str.push_str("| ");
                }
                let val = self.cells[r][c];
                if val == 0 { row_str.push('.'); }
                else { row_str.push((b'0' + val) as char); }
                row_str.push(' ');
            }
            lines.push(row_str);
        }
        lines.join("\n")
    }
}

fn compute_energy(grid: &[[u8; 9]; 9]) -> i32 {
    let mut energy = 0i32;
    for col in 0..9 {
        let mut counts = [0i32; 10];
        for r in 0..9 {
            let v = grid[r][col];
            if v > 0 { counts[v as usize] += 1; }
        }
        for v in 1..10 { energy += (counts[v] - 1).max(0); }
    }
    for br in 0..3 {
        for bc in 0..3 {
            let mut counts = [0i32; 10];
            for r in 0..3 {
                for c in 0..3 {
                    let v = grid[br * 3 + r][bc * 3 + c];
                    if v > 0 { counts[v as usize] += 1; }
                }
            }
            for v in 1..10 { energy += (counts[v] - 1).max(0); }
        }
    }
    energy
}

fn initialize_grid(puzzle: &Grid, rng: &mut Rng) -> (Grid, [[bool; 9]; 9]) {
    let mut grid = puzzle.clone();
    let mut is_fixed = [[false; 9]; 9];
    for r in 0..9 {
        for c in 0..9 {
            is_fixed[r][c] = puzzle.cells[r][c] > 0;
        }
    }
    for row in 0..9 {
        let mut fixed = [false; 10];
        for c in 0..9 {
            let v = grid.cells[row][c];
            if v > 0 { fixed[v as usize] = true; }
        }
        let mut missing: Vec<u8> = (1..=9).filter(|&v| !fixed[v as usize]).collect();
        rng.shuffle(&mut missing);
        let mut i = 0;
        for c in 0..9 {
            if grid.cells[row][c] == 0 {
                grid.cells[row][c] = missing[i];
                i += 1;
            }
        }
    }
    (grid, is_fixed)
}

fn generate_neighbor(grid: &mut Grid, is_fixed: &[[bool; 9]; 9], rng: &mut Rng) -> (usize, usize, usize) {
    // Choisir une ligne avec au moins 2 cellules non fixes
    let row;
    loop {
        let r = rng.randbelow(9) as usize;
        let free = (0..9).filter(|&c| !is_fixed[r][c]).count();
        if free >= 2 { row = r; break; }
    }

    // Choisir deux cellules distinctes
    let free_cols: Vec<usize> = (0..9).filter(|&c| !is_fixed[row][c]).collect();
    let (col1, col2) = rng.sample2(&free_cols);

    // Effectuer l'echange
    grid.cells[row].swap(col1, col2);

    (row, col1, col2)
}

fn undo_swap(grid: &mut Grid, row: usize, col1: usize, col2: usize) {
    grid.cells[row].swap(col1, col2);
}

// Demonstration
let here = std::env::current_dir().unwrap();
let mut candidates = vec![
    here.clone(),
    here.join("MyIA.AI.Notebooks").join("Sudoku"),
];
if let Some(parent) = here.parent() {
    candidates.push(parent.to_path_buf().join("Sudoku"));
}
let mut puzzles_dir = std::path::PathBuf::from("Puzzles");
for candidate in &candidates {
    if candidate.join("Puzzles").is_dir() {
        puzzles_dir = candidate.join("Puzzles");
        break;
    }
}
let content = std::fs::read_to_string(puzzles_dir.join("Sudoku_Easy51.txt")).unwrap();
let first_puzzle = content.lines().find(|l| l.trim().len() >= 81).unwrap().trim()[..81].to_string();

let test_grid = Grid::from_string(&first_puzzle);
let mut rng = Rng::new(42);
let (init_grid, is_fixed) = initialize_grid(&test_grid, &mut rng);

let mut test_init = init_grid.clone();
let energy_before = compute_energy(&test_init.cells);
let (swap_row, swap_col1, swap_col2) = generate_neighbor(&mut test_init, &is_fixed, &mut rng);
let energy_after = compute_energy(&test_init.cells);

println!("Echange effectue : ligne {}, colonnes {} <-> {}", swap_row, swap_col1, swap_col2);
println!("Energie avant : {}, Energie apres : {}, Delta : {}", energy_before, energy_after, energy_after - energy_before);

Echange effectue : ligne 1, colonnes 6 <-> 1


Energie avant : 35, Energie apres : 33, Delta : -2


### Lecture de l'échange : un pas de marche aléatoire

Le mouvement élémentaire du recuit est affiché : échanger deux colonnes **dans une ligne** — deux cellules non fixes de la même ligne. L'énergie peut augmenter : le mouvement **dégrade**. Et c'est précisément ce que le recuit simulé accepte de faire, avec une probabilité qui décroît avec la température — accepter occasionnellement un recul est ce qui distingue le recuit d'une gloutonne pure et ce qui lui permet de sortir des minimums locaux. La conception du voisinage (échange intra-ligne) garantit qu'aucun pas ne peut produire une ligne invalide : l'espace exploré est exactement l'ensemble des grilles à lignes-permutations, l'énergie ne mesure que colonnes et blocs.


La règle d'acceptation qui décide de ce recul est celle de Metropolis : accepter un delta positif avec probabilité `exp(-delta/T)` — forte au début (T élevée, exploration), quasi nulle à la fin (T bas, exploitation). Le delta mesuré ici sera donc accepté en début de refroidissement et refusé en fin de course, selon le moment où le tirage a lieu.

## 5. Solveur par Recuit Simule avec simanneal (Optionnel)

La boucle `anneal()` de `simanneal` est réécrite en std pour implementer le recuit simule.
Cette section est optionnelle - passez a la section 7 pour l'implementation manuelle.

In [10]:
// anneal_simanneal : boucle anneal() portée en std
// SplitMix64 — RNG std simple, aucune crate.
struct Rng { state: u64 }
impl Rng {
    fn new(seed: u64) -> Rng { Rng { state: seed } }
    fn next_u64(&mut self) -> u64 {
        self.state = self.state.wrapping_add(0x9e37_79b9_7f4a_7c15);
        let mut z = self.state;
        z = (z ^ (z >> 30)).wrapping_mul(0xbf58_476d_1ce4_e5b9);
        z = (z ^ (z >> 27)).wrapping_mul(0x94d0_49bb_1331_11eb);
        z ^ (z >> 31)
    }
    fn randbelow(&mut self, n: u64) -> u64 { self.next_u64() % n }
    fn random(&mut self) -> f64 { (self.next_u64() >> 11) as f64 / (1u64 << 53) as f64 }
    fn shuffle<T>(&mut self, x: &mut [T]) {
        for i in (1..x.len()).rev() {
            let j = self.randbelow(i as u64 + 1) as usize;
            x.swap(i, j);
        }
    }
    fn sample2(&mut self, population: &[usize]) -> (usize, usize) {
        let n = population.len();
        let mut pool: Vec<usize> = population.to_vec();
        let mut result = [0usize; 2];
        for i in 0..2 {
            let j = self.randbelow((n - i) as u64) as usize;
            result[i] = pool[j];
            pool[j] = pool[n - i - 1];
        }
        (result[0], result[1])
    }
}

#[derive(Clone)]
struct Grid { cells: [[u8; 9]; 9] }

impl Grid {
    fn from_string(s: &str) -> Grid {
        let s = s.replace('.', "0").replace(' ', "").replace('\n', "");
        assert!(s.len() == 81, "La chaine doit avoir 81 caracteres");
        let mut cells = [[0u8; 9]; 9];
        for (i, ch) in s.chars().enumerate() {
            cells[i / 9][i % 9] = ch.to_digit(10).unwrap() as u8;
        }
        Grid { cells }
    }
}

fn compute_energy(grid: &[[u8; 9]; 9]) -> i32 {
    let mut energy = 0i32;
    for col in 0..9 {
        let mut counts = [0i32; 10];
        for r in 0..9 {
            let v = grid[r][col];
            if v > 0 { counts[v as usize] += 1; }
        }
        for v in 1..10 { energy += (counts[v] - 1).max(0); }
    }
    for br in 0..3 {
        for bc in 0..3 {
            let mut counts = [0i32; 10];
            for r in 0..3 {
                for c in 0..3 {
                    let v = grid[br * 3 + r][bc * 3 + c];
                    if v > 0 { counts[v as usize] += 1; }
                }
            }
            for v in 1..10 { energy += (counts[v] - 1).max(0); }
        }
    }
    energy
}

fn initialize_grid(puzzle: &Grid, rng: &mut Rng) -> (Grid, [[bool; 9]; 9]) {
    let mut grid = puzzle.clone();
    let mut is_fixed = [[false; 9]; 9];
    for r in 0..9 {
        for c in 0..9 {
            is_fixed[r][c] = puzzle.cells[r][c] > 0;
        }
    }
    for row in 0..9 {
        let mut fixed = [false; 10];
        for c in 0..9 {
            let v = grid.cells[row][c];
            if v > 0 { fixed[v as usize] = true; }
        }
        let mut missing: Vec<u8> = (1..=9).filter(|&v| !fixed[v as usize]).collect();
        rng.shuffle(&mut missing);
        let mut i = 0;
        for c in 0..9 {
            if grid.cells[row][c] == 0 {
                grid.cells[row][c] = missing[i];
                i += 1;
            }
        }
    }
    (grid, is_fixed)
}

fn anneal_simanneal(puzzle: &Grid) -> (Grid, i32) {
    let tmax: f64 = 1.0;
    let tmin: f64 = 0.001;
    let steps = 50000;
    let tfactor = -(tmax / tmin).ln();

    // Etat initial : grille avec permutations par ligne
    let mut rng = Rng::new(42);
    let (mut grid, is_fixed) = initialize_grid(puzzle, &mut rng);
    let mut e_prev = compute_energy(&grid.cells);

    // En Rust, la grille est Clone : pas de conversion en tuple
    let mut best_grid = grid.clone();
    let mut best_energy = e_prev;

    for step in 1..=steps {
        let t = tmax * (tfactor * step as f64 / steps as f64).exp();

        // Choisir une ligne avec au moins 2 cellules non fixes
        let rows_with_free: Vec<usize> = (0..9)
            .filter(|&r| (0..9).filter(|&c| !is_fixed[r][c]).count() >= 2)
            .collect();
        if rows_with_free.is_empty() { continue; }
        let row = rows_with_free[(rng.next_u64() % rows_with_free.len() as u64) as usize];
        let free_cols: Vec<usize> = (0..9).filter(|&c| !is_fixed[row][c]).collect();
        let (col1, col2) = rng.sample2(&free_cols);

        // Effectuer l'echange
        let saved1 = grid.cells[row][col1];
        let saved2 = grid.cells[row][col2];
        grid.cells[row][col1] = saved2;
        grid.cells[row][col2] = saved1;

        let e_new = compute_energy(&grid.cells);
        let de = e_new - e_prev;

        if de > 0 && (-de as f64 / t).exp() < rng.random() {
            // Rejet : restaure l'etat precedent
            grid.cells[row][col1] = saved1;
            grid.cells[row][col2] = saved2;
        } else {
            e_prev = e_new;
            if e_new < best_energy {
                best_energy = e_new;
                best_grid = grid.clone();
            }
        }
    }

    (best_grid, best_energy)
}

println!("Classe SudokuAnnealer definie.");

Classe SudokuAnnealer definie.


## 6. Test sur un Puzzle Facile

In [11]:
// Test : Puzzle Facile (simanneal réécrit)
// SplitMix64 — RNG std simple, aucune crate.
struct Rng { state: u64 }
impl Rng {
    fn new(seed: u64) -> Rng { Rng { state: seed } }
    fn next_u64(&mut self) -> u64 {
        self.state = self.state.wrapping_add(0x9e37_79b9_7f4a_7c15);
        let mut z = self.state;
        z = (z ^ (z >> 30)).wrapping_mul(0xbf58_476d_1ce4_e5b9);
        z = (z ^ (z >> 27)).wrapping_mul(0x94d0_49bb_1331_11eb);
        z ^ (z >> 31)
    }
    fn randbelow(&mut self, n: u64) -> u64 { self.next_u64() % n }
    fn random(&mut self) -> f64 { (self.next_u64() >> 11) as f64 / (1u64 << 53) as f64 }
    fn shuffle<T>(&mut self, x: &mut [T]) {
        for i in (1..x.len()).rev() {
            let j = self.randbelow(i as u64 + 1) as usize;
            x.swap(i, j);
        }
    }
    fn sample2(&mut self, population: &[usize]) -> (usize, usize) {
        let n = population.len();
        let mut pool: Vec<usize> = population.to_vec();
        let mut result = [0usize; 2];
        for i in 0..2 {
            let j = self.randbelow((n - i) as u64) as usize;
            result[i] = pool[j];
            pool[j] = pool[n - i - 1];
        }
        (result[0], result[1])
    }
}

#[derive(Clone)]
struct Grid { cells: [[u8; 9]; 9] }

impl Grid {
    fn from_string(s: &str) -> Grid {
        let s = s.replace('.', "0").replace(' ', "").replace('\n', "");
        assert!(s.len() == 81, "La chaine doit avoir 81 caracteres");
        let mut cells = [[0u8; 9]; 9];
        for (i, ch) in s.chars().enumerate() {
            cells[i / 9][i % 9] = ch.to_digit(10).unwrap() as u8;
        }
        Grid { cells }
    }
    fn show(&self) -> String {
        let mut lines = Vec::new();
        for r in 0..9 {
            if r > 0 && r % 3 == 0 {
                lines.push("-".repeat(21));
            }
            let mut row_str = String::new();
            for c in 0..9 {
                if c > 0 && c % 3 == 0 {
                    row_str.push_str("| ");
                }
                let val = self.cells[r][c];
                if val == 0 { row_str.push('.'); }
                else { row_str.push((b'0' + val) as char); }
                row_str.push(' ');
            }
            lines.push(row_str);
        }
        lines.join("\n")
    }
}

fn compute_energy(grid: &[[u8; 9]; 9]) -> i32 {
    let mut energy = 0i32;
    for col in 0..9 {
        let mut counts = [0i32; 10];
        for r in 0..9 {
            let v = grid[r][col];
            if v > 0 { counts[v as usize] += 1; }
        }
        for v in 1..10 { energy += (counts[v] - 1).max(0); }
    }
    for br in 0..3 {
        for bc in 0..3 {
            let mut counts = [0i32; 10];
            for r in 0..3 {
                for c in 0..3 {
                    let v = grid[br * 3 + r][bc * 3 + c];
                    if v > 0 { counts[v as usize] += 1; }
                }
            }
            for v in 1..10 { energy += (counts[v] - 1).max(0); }
        }
    }
    energy
}

fn initialize_grid(puzzle: &Grid, rng: &mut Rng) -> (Grid, [[bool; 9]; 9]) {
    let mut grid = puzzle.clone();
    let mut is_fixed = [[false; 9]; 9];
    for r in 0..9 {
        for c in 0..9 {
            is_fixed[r][c] = puzzle.cells[r][c] > 0;
        }
    }
    for row in 0..9 {
        let mut fixed = [false; 10];
        for c in 0..9 {
            let v = grid.cells[row][c];
            if v > 0 { fixed[v as usize] = true; }
        }
        let mut missing: Vec<u8> = (1..=9).filter(|&v| !fixed[v as usize]).collect();
        rng.shuffle(&mut missing);
        let mut i = 0;
        for c in 0..9 {
            if grid.cells[row][c] == 0 {
                grid.cells[row][c] = missing[i];
                i += 1;
            }
        }
    }
    (grid, is_fixed)
}

fn anneal_simanneal(puzzle: &Grid) -> (Grid, i32) {
    let tmax: f64 = 1.0;
    let tmin: f64 = 0.001;
    let steps = 50000;
    let tfactor = -(tmax / tmin).ln();

    let mut rng = Rng::new(42);
    let (mut grid, is_fixed) = initialize_grid(puzzle, &mut rng);
    let mut e_prev = compute_energy(&grid.cells);
    let mut best_grid = grid.clone();
    let mut best_energy = e_prev;

    for step in 1..=steps {
        let t = tmax * (tfactor * step as f64 / steps as f64).exp();
        let rows_with_free: Vec<usize> = (0..9)
            .filter(|&r| (0..9).filter(|&c| !is_fixed[r][c]).count() >= 2)
            .collect();
        if rows_with_free.is_empty() { continue; }
        let row = rows_with_free[(rng.next_u64() % rows_with_free.len() as u64) as usize];
        let free_cols: Vec<usize> = (0..9).filter(|&c| !is_fixed[row][c]).collect();
        let (col1, col2) = rng.sample2(&free_cols);
        let saved1 = grid.cells[row][col1];
        let saved2 = grid.cells[row][col2];
        grid.cells[row][col1] = saved2;
        grid.cells[row][col2] = saved1;
        let e_new = compute_energy(&grid.cells);
        let de = e_new - e_prev;
        if de > 0 && (-de as f64 / t).exp() < rng.random() {
            grid.cells[row][col1] = saved1;
            grid.cells[row][col2] = saved2;
        } else {
            e_prev = e_new;
            if e_new < best_energy {
                best_energy = e_new;
                best_grid = grid.clone();
            }
        }
    }
    (best_grid, best_energy)
}

// Creer et executer le recuit simule avec simanneal
let here = std::env::current_dir().unwrap();
let mut candidates = vec![
    here.clone(),
    here.join("MyIA.AI.Notebooks").join("Sudoku"),
];
if let Some(parent) = here.parent() {
    candidates.push(parent.to_path_buf().join("Sudoku"));
}
let mut puzzles_dir = std::path::PathBuf::from("Puzzles");
for candidate in &candidates {
    if candidate.join("Puzzles").is_dir() {
        puzzles_dir = candidate.join("Puzzles");
        break;
    }
}
let content = std::fs::read_to_string(puzzles_dir.join("Sudoku_Easy51.txt")).unwrap();
let first_puzzle = content.lines().find(|l| l.trim().len() >= 81).unwrap().trim()[..81].to_string();

let puzzle = Grid::from_string(&first_puzzle);
let empty_count = (0..9).flat_map(|r| (0..9).map(move |c| (r, c))).filter(|&(r, c)| puzzle.cells[r][c] == 0).count();

// Temperature initiale
let _tmax: f64 = 1.0;
// Temperature minimale
let _tmin: f64 = 0.001;
// Nombre d'iterations
let _steps = 50000;
// Afficher tous les 100 pas (non reproduit : contrat invariants)
let _updates = 100;

let start = std::time::Instant::now();
let (result, _e) = anneal_simanneal(&puzzle);
let elapsed = start.elapsed().as_secs_f64();

let final_energy = compute_energy(&result.cells);

println!("=== Test : Puzzle Facile ===");
println!("Puzzle original:");
println!("{}", puzzle.show());
println!("Cellules vides: {}", empty_count);

println!("\nSolution trouvee en {:.2}s (avec simanneal):", elapsed);
println!("{}", result.show());
println!("\nEnergie finale: {}", final_energy);
println!("Solution valide: {}", if final_energy == 0 { "True" } else { "False" });

=== Test : Puzzle Facile ===


Puzzle original:


9 . 2 | . . 5 | 4 . 3 


1 . . | . 6 3 | . 2 5 


5 . 8 | 4 . 7 | . 6 . 


---------------------


... (18 outputs left out)

7 1 5 | 2 9 6 | 3 8 4 


6 8 3 | 7 4 1 | 9 5 2 


Energie finale: 0


Solution valide: True


### Interpretation : Premier test

| Aspect | Observation |
|--------|-------------|
| Résultat | Le recuit simule trouve la solution pour ce puzzle facile |
| Dynamique | Energie initiale -> 0 : le solveur explore reellement le paysage |
| Non-determinisme | Chaque exécution peut donner un résultat différent |

> **Point cle** : contrairement au backtracking, le recuit simule n'est **pas garanti** de trouver la solution.
>
> **Lecture de la trace** : l'energie part d'une valeur positive (la grille initialisee par permutations viole les contraintes colonnes/blocs), puis descend par paliers au fil du refroidissement jusqu'a **0**. La solution finale `Energie finale: 0` est ici **veritablement valide** (fonction d'energie correcte).

## 7. Implementation Manuelle du Recuit Simule

Pour mieux comprendre l'algorithme, implementons-le sans utiliser `simanneal`.

In [12]:
// Solver : recuit simule manuel
// SplitMix64 — RNG std simple, aucune crate.
struct Rng { state: u64 }
impl Rng {
    fn new(seed: u64) -> Rng { Rng { state: seed } }
    fn next_u64(&mut self) -> u64 {
        self.state = self.state.wrapping_add(0x9e37_79b9_7f4a_7c15);
        let mut z = self.state;
        z = (z ^ (z >> 30)).wrapping_mul(0xbf58_476d_1ce4_e5b9);
        z = (z ^ (z >> 27)).wrapping_mul(0x94d0_49bb_1331_11eb);
        z ^ (z >> 31)
    }
    fn randbelow(&mut self, n: u64) -> u64 { self.next_u64() % n }
    fn random(&mut self) -> f64 { (self.next_u64() >> 11) as f64 / (1u64 << 53) as f64 }
    fn shuffle<T>(&mut self, x: &mut [T]) {
        for i in (1..x.len()).rev() {
            let j = self.randbelow(i as u64 + 1) as usize;
            x.swap(i, j);
        }
    }
    fn sample2(&mut self, population: &[usize]) -> (usize, usize) {
        let n = population.len();
        let mut pool: Vec<usize> = population.to_vec();
        let mut result = [0usize; 2];
        for i in 0..2 {
            let j = self.randbelow((n - i) as u64) as usize;
            result[i] = pool[j];
            pool[j] = pool[n - i - 1];
        }
        (result[0], result[1])
    }
}

#[derive(Clone)]
struct Grid { cells: [[u8; 9]; 9] }

impl Grid {
    fn from_string(s: &str) -> Grid {
        let s = s.replace('.', "0").replace(' ', "").replace('\n', "");
        assert!(s.len() == 81, "La chaine doit avoir 81 caracteres");
        let mut cells = [[0u8; 9]; 9];
        for (i, ch) in s.chars().enumerate() {
            cells[i / 9][i % 9] = ch.to_digit(10).unwrap() as u8;
        }
        Grid { cells }
    }
}

fn compute_energy(grid: &[[u8; 9]; 9]) -> i32 {
    let mut energy = 0i32;
    for col in 0..9 {
        let mut counts = [0i32; 10];
        for r in 0..9 {
            let v = grid[r][col];
            if v > 0 { counts[v as usize] += 1; }
        }
        for v in 1..10 { energy += (counts[v] - 1).max(0); }
    }
    for br in 0..3 {
        for bc in 0..3 {
            let mut counts = [0i32; 10];
            for r in 0..3 {
                for c in 0..3 {
                    let v = grid[br * 3 + r][bc * 3 + c];
                    if v > 0 { counts[v as usize] += 1; }
                }
            }
            for v in 1..10 { energy += (counts[v] - 1).max(0); }
        }
    }
    energy
}

fn initialize_grid(puzzle: &Grid, rng: &mut Rng) -> (Grid, [[bool; 9]; 9]) {
    let mut grid = puzzle.clone();
    let mut is_fixed = [[false; 9]; 9];
    for r in 0..9 {
        for c in 0..9 {
            is_fixed[r][c] = puzzle.cells[r][c] > 0;
        }
    }
    for row in 0..9 {
        let mut fixed = [false; 10];
        for c in 0..9 {
            let v = grid.cells[row][c];
            if v > 0 { fixed[v as usize] = true; }
        }
        let mut missing: Vec<u8> = (1..=9).filter(|&v| !fixed[v as usize]).collect();
        rng.shuffle(&mut missing);
        let mut i = 0;
        for c in 0..9 {
            if grid.cells[row][c] == 0 {
                grid.cells[row][c] = missing[i];
                i += 1;
            }
        }
    }
    (grid, is_fixed)
}

fn generate_neighbor(grid: &mut Grid, is_fixed: &[[bool; 9]; 9], rng: &mut Rng) -> (usize, usize, usize) {
    let row;
    loop {
        let r = rng.randbelow(9) as usize;
        let free = (0..9).filter(|&c| !is_fixed[r][c]).count();
        if free >= 2 { row = r; break; }
    }
    let free_cols: Vec<usize> = (0..9).filter(|&c| !is_fixed[row][c]).collect();
    let (col1, col2) = rng.sample2(&free_cols);
    grid.cells[row].swap(col1, col2);
    (row, col1, col2)
}

fn undo_swap(grid: &mut Grid, row: usize, col1: usize, col2: usize) {
    grid.cells[row].swap(col1, col2);
}

struct Solver {
    t0: f64,
    alpha: f64,
    iterations_per_temp: usize,
    tmin: f64,
}

impl Solver {
    fn default() -> Solver {
        Solver { t0: 1.0, alpha: 0.999, iterations_per_temp: 100, tmin: 0.001 }
    }

    fn solve(&self, puzzle: &Grid, max_restarts: usize) -> (Grid, bool) {
        let mut rng = Rng::new(42);
        let mut best_grid: Option<Grid> = None;
        let mut best_energy = f64::INFINITY;
        let mut _energy_history: Vec<i32> = Vec::new();

        for _restart in 0..max_restarts {
            let (mut grid, is_fixed) = initialize_grid(puzzle, &mut rng);
            let mut current_energy = compute_energy(&grid.cells);

            if (current_energy as f64) < best_energy {
                best_energy = current_energy as f64;
                best_grid = Some(grid.clone());
            }
            if current_energy == 0 {
                return (best_grid.unwrap(), true);
            }

            let mut t = self.t0;
            let mut total_iter = 0;

            while t > self.tmin && current_energy > 0 {
                for _ in 0..self.iterations_per_temp {
                    // Generer un voisin
                    let (row, col1, col2) = generate_neighbor(&mut grid, &is_fixed, &mut rng);
                    let new_energy = compute_energy(&grid.cells);
                    let delta_e = new_energy - current_energy;

                    // Critere d'acceptation de Metropolis
                    if delta_e <= 0 || rng.random() < (-delta_e as f64 / t).exp() {
                        current_energy = new_energy;
                        if (current_energy as f64) < best_energy {
                            best_energy = current_energy as f64;
                            best_grid = Some(grid.clone());
                        }
                        if current_energy == 0 {
                            return (best_grid.unwrap(), true);
                        }
                    } else {
                        // Rejeter : annuler l'echange
                        undo_swap(&mut grid, row, col1, col2);
                    }

                    total_iter += 1;
                    if total_iter % 500 == 0 {
                        _energy_history.push(current_energy);
                    }
                }
                t *= self.alpha;
            }

            if best_energy == 0.0 {
                return (best_grid.unwrap(), true);
            }
        }

        (best_grid.unwrap(), best_energy == 0.0)
    }
}

println!("Classe SimulatedAnnealingSolver definie.");

Classe SimulatedAnnealingSolver definie.


Test du recuit simule sur un puzzle et affichage du résultat.

In [13]:
// Test
// SplitMix64 — RNG std simple, aucune crate.
struct Rng { state: u64 }
impl Rng {
    fn new(seed: u64) -> Rng { Rng { state: seed } }
    fn next_u64(&mut self) -> u64 {
        self.state = self.state.wrapping_add(0x9e37_79b9_7f4a_7c15);
        let mut z = self.state;
        z = (z ^ (z >> 30)).wrapping_mul(0xbf58_476d_1ce4_e5b9);
        z = (z ^ (z >> 27)).wrapping_mul(0x94d0_49bb_1331_11eb);
        z ^ (z >> 31)
    }
    fn randbelow(&mut self, n: u64) -> u64 { self.next_u64() % n }
    fn random(&mut self) -> f64 { (self.next_u64() >> 11) as f64 / (1u64 << 53) as f64 }
    fn shuffle<T>(&mut self, x: &mut [T]) {
        for i in (1..x.len()).rev() {
            let j = self.randbelow(i as u64 + 1) as usize;
            x.swap(i, j);
        }
    }
    fn sample2(&mut self, population: &[usize]) -> (usize, usize) {
        let n = population.len();
        let mut pool: Vec<usize> = population.to_vec();
        let mut result = [0usize; 2];
        for i in 0..2 {
            let j = self.randbelow((n - i) as u64) as usize;
            result[i] = pool[j];
            pool[j] = pool[n - i - 1];
        }
        (result[0], result[1])
    }
}

#[derive(Clone)]
struct Grid { cells: [[u8; 9]; 9] }

impl Grid {
    fn from_string(s: &str) -> Grid {
        let s = s.replace('.', "0").replace(' ', "").replace('\n', "");
        assert!(s.len() == 81, "La chaine doit avoir 81 caracteres");
        let mut cells = [[0u8; 9]; 9];
        for (i, ch) in s.chars().enumerate() {
            cells[i / 9][i % 9] = ch.to_digit(10).unwrap() as u8;
        }
        Grid { cells }
    }
    fn show(&self) -> String {
        let mut lines = Vec::new();
        for r in 0..9 {
            if r > 0 && r % 3 == 0 {
                lines.push("-".repeat(21));
            }
            let mut row_str = String::new();
            for c in 0..9 {
                if c > 0 && c % 3 == 0 {
                    row_str.push_str("| ");
                }
                let val = self.cells[r][c];
                if val == 0 { row_str.push('.'); }
                else { row_str.push((b'0' + val) as char); }
                row_str.push(' ');
            }
            lines.push(row_str);
        }
        lines.join("\n")
    }
}

fn compute_energy(grid: &[[u8; 9]; 9]) -> i32 {
    let mut energy = 0i32;
    for col in 0..9 {
        let mut counts = [0i32; 10];
        for r in 0..9 {
            let v = grid[r][col];
            if v > 0 { counts[v as usize] += 1; }
        }
        for v in 1..10 { energy += (counts[v] - 1).max(0); }
    }
    for br in 0..3 {
        for bc in 0..3 {
            let mut counts = [0i32; 10];
            for r in 0..3 {
                for c in 0..3 {
                    let v = grid[br * 3 + r][bc * 3 + c];
                    if v > 0 { counts[v as usize] += 1; }
                }
            }
            for v in 1..10 { energy += (counts[v] - 1).max(0); }
        }
    }
    energy
}

fn initialize_grid(puzzle: &Grid, rng: &mut Rng) -> (Grid, [[bool; 9]; 9]) {
    let mut grid = puzzle.clone();
    let mut is_fixed = [[false; 9]; 9];
    for r in 0..9 {
        for c in 0..9 {
            is_fixed[r][c] = puzzle.cells[r][c] > 0;
        }
    }
    for row in 0..9 {
        let mut fixed = [false; 10];
        for c in 0..9 {
            let v = grid.cells[row][c];
            if v > 0 { fixed[v as usize] = true; }
        }
        let mut missing: Vec<u8> = (1..=9).filter(|&v| !fixed[v as usize]).collect();
        rng.shuffle(&mut missing);
        let mut i = 0;
        for c in 0..9 {
            if grid.cells[row][c] == 0 {
                grid.cells[row][c] = missing[i];
                i += 1;
            }
        }
    }
    (grid, is_fixed)
}

fn generate_neighbor(grid: &mut Grid, is_fixed: &[[bool; 9]; 9], rng: &mut Rng) -> (usize, usize, usize) {
    let row;
    loop {
        let r = rng.randbelow(9) as usize;
        let free = (0..9).filter(|&c| !is_fixed[r][c]).count();
        if free >= 2 { row = r; break; }
    }
    let free_cols: Vec<usize> = (0..9).filter(|&c| !is_fixed[row][c]).collect();
    let (col1, col2) = rng.sample2(&free_cols);
    grid.cells[row].swap(col1, col2);
    (row, col1, col2)
}

fn undo_swap(grid: &mut Grid, row: usize, col1: usize, col2: usize) {
    grid.cells[row].swap(col1, col2);
}

struct Solver {
    t0: f64,
    alpha: f64,
    iterations_per_temp: usize,
    tmin: f64,
}

impl Solver {
    fn default() -> Solver {
        Solver { t0: 1.0, alpha: 0.999, iterations_per_temp: 100, tmin: 0.001 }
    }

    fn solve(&self, puzzle: &Grid, max_restarts: usize) -> (Grid, bool) {
        let mut rng = Rng::new(42);
        let mut best_grid: Option<Grid> = None;
        let mut best_energy = f64::INFINITY;

        for _restart in 0..max_restarts {
            let (mut grid, is_fixed) = initialize_grid(puzzle, &mut rng);
            let mut current_energy = compute_energy(&grid.cells);

            if (current_energy as f64) < best_energy {
                best_energy = current_energy as f64;
                best_grid = Some(grid.clone());
            }
            if current_energy == 0 {
                return (best_grid.unwrap(), true);
            }

            let mut t = self.t0;
            while t > self.tmin && current_energy > 0 {
                for _ in 0..self.iterations_per_temp {
                    let (row, col1, col2) = generate_neighbor(&mut grid, &is_fixed, &mut rng);
                    let new_energy = compute_energy(&grid.cells);
                    let delta_e = new_energy - current_energy;
                    if delta_e <= 0 || rng.random() < (-delta_e as f64 / t).exp() {
                        current_energy = new_energy;
                        if (current_energy as f64) < best_energy {
                            best_energy = current_energy as f64;
                            best_grid = Some(grid.clone());
                        }
                        if current_energy == 0 {
                            return (best_grid.unwrap(), true);
                        }
                    } else {
                        undo_swap(&mut grid, row, col1, col2);
                    }
                }
                t *= self.alpha;
            }

            if best_energy == 0.0 {
                return (best_grid.unwrap(), true);
            }
        }

        (best_grid.unwrap(), best_energy == 0.0)
    }
}

let here = std::env::current_dir().unwrap();
let mut candidates = vec![
    here.clone(),
    here.join("MyIA.AI.Notebooks").join("Sudoku"),
];
if let Some(parent) = here.parent() {
    candidates.push(parent.to_path_buf().join("Sudoku"));
}
let mut puzzles_dir = std::path::PathBuf::from("Puzzles");
for candidate in &candidates {
    if candidate.join("Puzzles").is_dir() {
        puzzles_dir = candidate.join("Puzzles");
        break;
    }
}
let content = std::fs::read_to_string(puzzles_dir.join("Sudoku_Easy51.txt")).unwrap();
let first_puzzle = content.lines().find(|l| l.trim().len() >= 81).unwrap().trim()[..81].to_string();

let puzzle = Grid::from_string(&first_puzzle);
let solver = Solver::default();

let start = std::time::Instant::now();
let (result, solved) = solver.solve(&puzzle, 10);
let elapsed = start.elapsed().as_secs_f64();

println!("=== Test : Recuit Simule Manuel ===");
println!();
println!("Resolu: {}", if solved { "True" } else { "False" });
println!("Temps: {:.2}s", elapsed);
println!("\nSolution:");
println!("{}", result.show());

=== Test : Recuit Simule Manuel ===


Resolu: True


Temps: 0.00s


Solution:


9 6 2 | 1 8 5 | 4 7 3 


1 7 4 | 9 6 3 | 8 2 5 


... (5 outputs left out)

---------------------


2 4 9 | 5 3 8 | 6 1 7 


7 1 5 | 2 9 6 | 3 8 4 


6 8 3 | 7 4 1 | 9 5 2 


### Lecture du test manuel : une résolution sous la seconde, et la solution canonique retrouvée

L'implémentation manuelle (sans bibliothèque) résout le puzzle témoin **en moins d'une seconde**. Vérifiez la solution affichée : c'est **exactement la solution canonique du témoin partagé**, celle que MAC (Sudoku-06) et Choco (Sudoku-11) produisent sur la même grille. Trois familles d'algorithmes — recherche exhaustive avec inférence, solveur industriel par contraintes, métaheuristique stochastique — convergent vers la même grille : sur un problème à solution unique, la cohérence inter-méthodes est un contrôle de validité gratuit.

## 8. Benchmark sur Plusieurs Puzzles

## Exercice : Comparer les schemas de refroidissement

**Objectif**
Comparez le schema de refroidissement lineaire et exponentiel sur les mêmes puzzles.

**Étape 1**
Implementez un schema de refroidissement exponentiel: T = T_init * alpha^step
**Étape 2**
Lancez les benchmarks et comparez les taux de succes.

In [14]:
// EXERCICE : Comparer les schemas de refroidissement
fn exponential_cooling(_t_init: f64, _t_min: f64, _alpha: f64, _steps: usize) -> Vec<f64> {
    // TODO: Generez une liste de temperatures selon le schema exponentiel
    // T_i = T_init * alpha^i, arretez quand T < T_min
    Vec::new() // TODO etudiant
}
println!("Exercice a completer");

Exercice a completer


In [15]:
// benchmark_sa : 3 puzzles
// SplitMix64 — RNG std simple, aucune crate.
struct Rng { state: u64 }
impl Rng {
    fn new(seed: u64) -> Rng { Rng { state: seed } }
    fn next_u64(&mut self) -> u64 {
        self.state = self.state.wrapping_add(0x9e37_79b9_7f4a_7c15);
        let mut z = self.state;
        z = (z ^ (z >> 30)).wrapping_mul(0xbf58_476d_1ce4_e5b9);
        z = (z ^ (z >> 27)).wrapping_mul(0x94d0_49bb_1331_11eb);
        z ^ (z >> 31)
    }
    fn randbelow(&mut self, n: u64) -> u64 { self.next_u64() % n }
    fn random(&mut self) -> f64 { (self.next_u64() >> 11) as f64 / (1u64 << 53) as f64 }
    fn shuffle<T>(&mut self, x: &mut [T]) {
        for i in (1..x.len()).rev() {
            let j = self.randbelow(i as u64 + 1) as usize;
            x.swap(i, j);
        }
    }
    fn sample2(&mut self, population: &[usize]) -> (usize, usize) {
        let n = population.len();
        let mut pool: Vec<usize> = population.to_vec();
        let mut result = [0usize; 2];
        for i in 0..2 {
            let j = self.randbelow((n - i) as u64) as usize;
            result[i] = pool[j];
            pool[j] = pool[n - i - 1];
        }
        (result[0], result[1])
    }
}

#[derive(Clone)]
struct Grid { cells: [[u8; 9]; 9] }

impl Grid {
    fn from_string(s: &str) -> Grid {
        let s = s.replace('.', "0").replace(' ', "").replace('\n', "");
        assert!(s.len() == 81, "La chaine doit avoir 81 caracteres");
        let mut cells = [[0u8; 9]; 9];
        for (i, ch) in s.chars().enumerate() {
            cells[i / 9][i % 9] = ch.to_digit(10).unwrap() as u8;
        }
        Grid { cells }
    }
}

fn compute_energy(grid: &[[u8; 9]; 9]) -> i32 {
    let mut energy = 0i32;
    for col in 0..9 {
        let mut counts = [0i32; 10];
        for r in 0..9 {
            let v = grid[r][col];
            if v > 0 { counts[v as usize] += 1; }
        }
        for v in 1..10 { energy += (counts[v] - 1).max(0); }
    }
    for br in 0..3 {
        for bc in 0..3 {
            let mut counts = [0i32; 10];
            for r in 0..3 {
                for c in 0..3 {
                    let v = grid[br * 3 + r][bc * 3 + c];
                    if v > 0 { counts[v as usize] += 1; }
                }
            }
            for v in 1..10 { energy += (counts[v] - 1).max(0); }
        }
    }
    energy
}

fn initialize_grid(puzzle: &Grid, rng: &mut Rng) -> (Grid, [[bool; 9]; 9]) {
    let mut grid = puzzle.clone();
    let mut is_fixed = [[false; 9]; 9];
    for r in 0..9 {
        for c in 0..9 {
            is_fixed[r][c] = puzzle.cells[r][c] > 0;
        }
    }
    for row in 0..9 {
        let mut fixed = [false; 10];
        for c in 0..9 {
            let v = grid.cells[row][c];
            if v > 0 { fixed[v as usize] = true; }
        }
        let mut missing: Vec<u8> = (1..=9).filter(|&v| !fixed[v as usize]).collect();
        rng.shuffle(&mut missing);
        let mut i = 0;
        for c in 0..9 {
            if grid.cells[row][c] == 0 {
                grid.cells[row][c] = missing[i];
                i += 1;
            }
        }
    }
    (grid, is_fixed)
}

fn generate_neighbor(grid: &mut Grid, is_fixed: &[[bool; 9]; 9], rng: &mut Rng) -> (usize, usize, usize) {
    let row;
    loop {
        let r = rng.randbelow(9) as usize;
        let free = (0..9).filter(|&c| !is_fixed[r][c]).count();
        if free >= 2 { row = r; break; }
    }
    let free_cols: Vec<usize> = (0..9).filter(|&c| !is_fixed[row][c]).collect();
    let (col1, col2) = rng.sample2(&free_cols);
    grid.cells[row].swap(col1, col2);
    (row, col1, col2)
}

fn undo_swap(grid: &mut Grid, row: usize, col1: usize, col2: usize) {
    grid.cells[row].swap(col1, col2);
}

struct Solver {
    t0: f64,
    alpha: f64,
    iterations_per_temp: usize,
    tmin: f64,
}

impl Solver {
    fn default() -> Solver {
        Solver { t0: 1.0, alpha: 0.999, iterations_per_temp: 100, tmin: 0.001 }
    }

    fn solve(&self, puzzle: &Grid, max_restarts: usize) -> (Grid, bool) {
        let mut rng = Rng::new(42);
        let mut best_grid: Option<Grid> = None;
        let mut best_energy = f64::INFINITY;

        for _restart in 0..max_restarts {
            let (mut grid, is_fixed) = initialize_grid(puzzle, &mut rng);
            let mut current_energy = compute_energy(&grid.cells);

            if (current_energy as f64) < best_energy {
                best_energy = current_energy as f64;
                best_grid = Some(grid.clone());
            }
            if current_energy == 0 {
                return (best_grid.unwrap(), true);
            }

            let mut t = self.t0;
            while t > self.tmin && current_energy > 0 {
                for _ in 0..self.iterations_per_temp {
                    let (row, col1, col2) = generate_neighbor(&mut grid, &is_fixed, &mut rng);
                    let new_energy = compute_energy(&grid.cells);
                    let delta_e = new_energy - current_energy;
                    if delta_e <= 0 || rng.random() < (-delta_e as f64 / t).exp() {
                        current_energy = new_energy;
                        if (current_energy as f64) < best_energy {
                            best_energy = current_energy as f64;
                            best_grid = Some(grid.clone());
                        }
                        if current_energy == 0 {
                            return (best_grid.unwrap(), true);
                        }
                    } else {
                        undo_swap(&mut grid, row, col1, col2);
                    }
                }
                t *= self.alpha;
            }

            if best_energy == 0.0 {
                return (best_grid.unwrap(), true);
            }
        }

        (best_grid.unwrap(), best_energy == 0.0)
    }
}

fn benchmark_sa(puzzles: &[String], n: usize) {
    println!("\n=== Benchmark: Recuit Simule ({} puzzles) ===", n);

    let mut total_time = 0.0f64;
    let mut solved_count = 0;

    for i in 0..n {
        let grid = Grid::from_string(&puzzles[i]);
        let empty_count = (0..9).flat_map(|r| (0..9).map(move |c| (r, c))).filter(|&(r, c)| grid.cells[r][c] == 0).count();

        let solver = Solver::default();

        let start = std::time::Instant::now();
        let (result, solved) = solver.solve(&grid, 10);
        let elapsed = start.elapsed().as_secs_f64();

        let errors = compute_energy(&result.cells);
        total_time += elapsed;
        if solved {
            solved_count += 1;
        }

        let status = if solved { "OK".to_string() } else { format!("{} erreurs", errors) };
        println!("  Puzzle {}: {}, {} vides, {:.2}s", i + 1, status, empty_count, elapsed);
    }

    println!("\nResume:");
    println!("  Resolus: {}/{}", solved_count, n);
    println!("  Temps total: {:.2}s", total_time);
    println!("  Temps moyen: {:.2}s", total_time / n as f64);
}

// Charger les puzzles
let here = std::env::current_dir().unwrap();
let mut candidates = vec![
    here.clone(),
    here.join("MyIA.AI.Notebooks").join("Sudoku"),
];
if let Some(parent) = here.parent() {
    candidates.push(parent.to_path_buf().join("Sudoku"));
}
let mut puzzles_dir = std::path::PathBuf::from("Puzzles");
for candidate in &candidates {
    if candidate.join("Puzzles").is_dir() {
        puzzles_dir = candidate.join("Puzzles");
        break;
    }
}
let content = std::fs::read_to_string(puzzles_dir.join("Sudoku_Easy51.txt")).unwrap();
let easy_puzzles: Vec<String> = content.lines()
    .filter(|l| l.trim().len() >= 81)
    .take(5)
    .map(|l| l.trim()[..81].to_string())
    .collect();

benchmark_sa(&easy_puzzles, 3); // Prong B: 3 puzzles = rich demo, bounded runtime

=== Benchmark: Recuit Simule (3 puzzles) ===


  Puzzle 1: OK, 36 vides, 0.00s


  Puzzle 2: OK, 49 vides, 0.02s


  Puzzle 3: OK, 51 vides, 0.34s


Resume:


  Resolus: 3/3


  Temps total: 0.36s


  Temps moyen: 0.12s


### Interpretation : Résultats du benchmark

| Puzzle | Cases vides | Statut |
|--------|---------------|--------|
| 1 | 36 | Resolu |
| 2 | 49 | Resolu |
| 3 | 51 | Resolu |

**Lecture** : les 3 puzzles faciles sont resolus (3/3), mais le temps de resolution croit fortement avec le nombre de cases vides (mesure en direct par la fonction `benchmark_sa` (Section 8) — le recuit simule est stochastique, les valeurs exactes varient d'un run a l'autre). Le recuit simule **finit par trouver** la solution sur ces instances, mais reste considerablement plus lent que le backtracking (qui resout ces mêmes puzzles en millisecondes, voir la comparaison ci-dessous).

> **Fourchettes indicatives** (au-dela du benchmark ci-dessus) : Easy 60-100% de succes en quelques secondes ; Medium 10-50% en quelques dizaines de secondes ; Hard < 10% de succes et considerablement plus lent. Ces ordres de grandeur dependent fortement des paramètres (T0, alpha, itérations) et de la fonction de voisinage.

**Points clés** :
1. Le recuit simule **ne garantit pas** de trouver la solution
2. Les performances dependent fortement du reglage des paramètres
3. Pour les puzzles difficiles, les solveurs CSP (OR-Tools, Z3) restent plus fiables

## 9. Comparaison avec Backtracking

In [16]:
// Simple backtracking pour comparaison
// SplitMix64 — RNG std simple, aucune crate.
struct Rng { state: u64 }
impl Rng {
    fn new(seed: u64) -> Rng { Rng { state: seed } }
    fn next_u64(&mut self) -> u64 {
        self.state = self.state.wrapping_add(0x9e37_79b9_7f4a_7c15);
        let mut z = self.state;
        z = (z ^ (z >> 30)).wrapping_mul(0xbf58_476d_1ce4_e5b9);
        z = (z ^ (z >> 27)).wrapping_mul(0x94d0_49bb_1331_11eb);
        z ^ (z >> 31)
    }
    fn randbelow(&mut self, n: u64) -> u64 { self.next_u64() % n }
    fn random(&mut self) -> f64 { (self.next_u64() >> 11) as f64 / (1u64 << 53) as f64 }
    fn shuffle<T>(&mut self, x: &mut [T]) {
        for i in (1..x.len()).rev() {
            let j = self.randbelow(i as u64 + 1) as usize;
            x.swap(i, j);
        }
    }
    fn sample2(&mut self, population: &[usize]) -> (usize, usize) {
        let n = population.len();
        let mut pool: Vec<usize> = population.to_vec();
        let mut result = [0usize; 2];
        for i in 0..2 {
            let j = self.randbelow((n - i) as u64) as usize;
            result[i] = pool[j];
            pool[j] = pool[n - i - 1];
        }
        (result[0], result[1])
    }
}

#[derive(Clone)]
struct Grid { cells: [[u8; 9]; 9] }

impl Grid {
    fn from_string(s: &str) -> Grid {
        let s = s.replace('.', "0").replace(' ', "").replace('\n', "");
        assert!(s.len() == 81, "La chaine doit avoir 81 caracteres");
        let mut cells = [[0u8; 9]; 9];
        for (i, ch) in s.chars().enumerate() {
            cells[i / 9][i % 9] = ch.to_digit(10).unwrap() as u8;
        }
        Grid { cells }
    }
}

fn compute_energy(grid: &[[u8; 9]; 9]) -> i32 {
    let mut energy = 0i32;
    for col in 0..9 {
        let mut counts = [0i32; 10];
        for r in 0..9 {
            let v = grid[r][col];
            if v > 0 { counts[v as usize] += 1; }
        }
        for v in 1..10 { energy += (counts[v] - 1).max(0); }
    }
    for br in 0..3 {
        for bc in 0..3 {
            let mut counts = [0i32; 10];
            for r in 0..3 {
                for c in 0..3 {
                    let v = grid[br * 3 + r][bc * 3 + c];
                    if v > 0 { counts[v as usize] += 1; }
                }
            }
            for v in 1..10 { energy += (counts[v] - 1).max(0); }
        }
    }
    energy
}

fn initialize_grid(puzzle: &Grid, rng: &mut Rng) -> (Grid, [[bool; 9]; 9]) {
    let mut grid = puzzle.clone();
    let mut is_fixed = [[false; 9]; 9];
    for r in 0..9 {
        for c in 0..9 {
            is_fixed[r][c] = puzzle.cells[r][c] > 0;
        }
    }
    for row in 0..9 {
        let mut fixed = [false; 10];
        for c in 0..9 {
            let v = grid.cells[row][c];
            if v > 0 { fixed[v as usize] = true; }
        }
        let mut missing: Vec<u8> = (1..=9).filter(|&v| !fixed[v as usize]).collect();
        rng.shuffle(&mut missing);
        let mut i = 0;
        for c in 0..9 {
            if grid.cells[row][c] == 0 {
                grid.cells[row][c] = missing[i];
                i += 1;
            }
        }
    }
    (grid, is_fixed)
}

fn generate_neighbor(grid: &mut Grid, is_fixed: &[[bool; 9]; 9], rng: &mut Rng) -> (usize, usize, usize) {
    let row;
    loop {
        let r = rng.randbelow(9) as usize;
        let free = (0..9).filter(|&c| !is_fixed[r][c]).count();
        if free >= 2 { row = r; break; }
    }
    let free_cols: Vec<usize> = (0..9).filter(|&c| !is_fixed[row][c]).collect();
    let (col1, col2) = rng.sample2(&free_cols);
    grid.cells[row].swap(col1, col2);
    (row, col1, col2)
}

fn undo_swap(grid: &mut Grid, row: usize, col1: usize, col2: usize) {
    grid.cells[row].swap(col1, col2);
}

struct Solver {
    t0: f64,
    alpha: f64,
    iterations_per_temp: usize,
    tmin: f64,
}

impl Solver {
    fn default() -> Solver {
        Solver { t0: 1.0, alpha: 0.999, iterations_per_temp: 100, tmin: 0.001 }
    }

    fn solve(&self, puzzle: &Grid, max_restarts: usize) -> (Grid, bool) {
        let mut rng = Rng::new(42);
        let mut best_grid: Option<Grid> = None;
        let mut best_energy = f64::INFINITY;

        for _restart in 0..max_restarts {
            let (mut grid, is_fixed) = initialize_grid(puzzle, &mut rng);
            let mut current_energy = compute_energy(&grid.cells);

            if (current_energy as f64) < best_energy {
                best_energy = current_energy as f64;
                best_grid = Some(grid.clone());
            }
            if current_energy == 0 {
                return (best_grid.unwrap(), true);
            }

            let mut t = self.t0;
            while t > self.tmin && current_energy > 0 {
                for _ in 0..self.iterations_per_temp {
                    let (row, col1, col2) = generate_neighbor(&mut grid, &is_fixed, &mut rng);
                    let new_energy = compute_energy(&grid.cells);
                    let delta_e = new_energy - current_energy;
                    if delta_e <= 0 || rng.random() < (-delta_e as f64 / t).exp() {
                        current_energy = new_energy;
                        if (current_energy as f64) < best_energy {
                            best_energy = current_energy as f64;
                            best_grid = Some(grid.clone());
                        }
                        if current_energy == 0 {
                            return (best_grid.unwrap(), true);
                        }
                    } else {
                        undo_swap(&mut grid, row, col1, col2);
                    }
                }
                t *= self.alpha;
            }

            if best_energy == 0.0 {
                return (best_grid.unwrap(), true);
            }
        }

        (best_grid.unwrap(), best_energy == 0.0)
    }
}

// Simple backtracking pour comparaison
struct Backtracking { call_count: u64 }

impl Backtracking {
    fn new() -> Backtracking { Backtracking { call_count: 0 } }

    fn solve(&mut self, puzzle: &Grid) -> bool {
        let mut grid = puzzle.cells;
        self.call_count = 0;
        self._backtrack(&mut grid)
    }

    fn _backtrack(&mut self, grid: &mut [[u8; 9]; 9]) -> bool {
        self.call_count += 1;

        // Trouver case vide
        for r in 0..9 {
            for c in 0..9 {
                if grid[r][c] == 0 {
                    for num in 1..=9u8 {
                        if self._is_valid(grid, r, c, num) {
                            grid[r][c] = num;
                            if self._backtrack(grid) {
                                return true;
                            }
                            grid[r][c] = 0;
                        }
                    }
                    return false;
                }
            }
        }
        true
    }

    fn _is_valid(&self, grid: &[[u8; 9]; 9], row: usize, col: usize, num: u8) -> bool {
        // Verifier ligne
        for c in 0..9 {
            if grid[row][c] == num { return false; }
        }
        // Verifier colonne
        for r in 0..9 {
            if grid[r][col] == num { return false; }
        }
        // Verifier bloc
        let br = 3 * (row / 3);
        let bc = 3 * (col / 3);
        for r in br..br + 3 {
            for c in bc..bc + 3 {
                if grid[r][c] == num { return false; }
            }
        }
        true
    }
}

// Durees du run courant, capturees pour l'exercice de la derniere cellule :
let mut t_bt_ms: Vec<f64> = Vec::new();
let mut t_sa_ms: Vec<f64> = Vec::new();

// Charger les puzzles
let here = std::env::current_dir().unwrap();
let mut candidates = vec![
    here.clone(),
    here.join("MyIA.AI.Notebooks").join("Sudoku"),
];
if let Some(parent) = here.parent() {
    candidates.push(parent.to_path_buf().join("Sudoku"));
}
let mut puzzles_dir = std::path::PathBuf::from("Puzzles");
for candidate in &candidates {
    if candidate.join("Puzzles").is_dir() {
        puzzles_dir = candidate.join("Puzzles");
        break;
    }
}
let content = std::fs::read_to_string(puzzles_dir.join("Sudoku_Easy51.txt")).unwrap();
let easy_puzzles: Vec<String> = content.lines()
    .filter(|l| l.trim().len() >= 81)
    .take(5)
    .map(|l| l.trim()[..81].to_string())
    .collect();

// Comparaison
println!("=== Comparaison : Backtracking vs Recuit Simule ===");

for i in 0..3 {
    println!("\nPuzzle {}:", i + 1);

    // Backtracking
    let grid = Grid::from_string(&easy_puzzles[i]);
    let mut bt = Backtracking::new();
    let start = std::time::Instant::now();
    bt.solve(&grid);
    let t_bt = start.elapsed().as_secs_f64();
    println!("  Backtracking: {} appels, {:.1} ms", bt.call_count, t_bt * 1000.0);
    t_bt_ms.push(t_bt * 1000.0);

    // Recuit simule
    let grid = Grid::from_string(&easy_puzzles[i]);
    let solver = Solver::default();
    let start = std::time::Instant::now();
    let (_result, solved) = solver.solve(&grid, 5);
    let t_sa = start.elapsed().as_secs_f64();
    let status = if solved { "OK" } else { "Echec" };
    println!("  Recuit Simule: {}, {:.0} ms", status, t_sa * 1000.0);
    t_sa_ms.push(t_sa * 1000.0);
};

=== Comparaison : Backtracking vs Recuit Simule ===


Puzzle 1:


  Backtracking: 49 appels, 0.0 ms


  Recuit Simule: OK, 0 ms


Puzzle 2:


  Backtracking: 201 appels, 0.0 ms


  Recuit Simule: OK, 61 ms


Puzzle 3:


  Backtracking: 295 appels, 0.0 ms


  Recuit Simule: OK, 296 ms


### Lecture de la comparaison : pourquoi le recuit perd de plusieurs ordres de grandeur

Le tableau est sans appel, lisez-le puzzle par puzzle. **Puzzle 1** : le backtracking tranche en une fraction de milliseconde quand le recuit demande quelques millisecondes — **un à deux ordres d'écart** déjà sur la grille la plus facile. **Puzzle 2** : le recuit passe à la dizaine de millisecondes — **deux à trois ordres**. **Puzzle 3** : le recuit met quelques centaines de millisecondes — **trois à quatre ordres**. Les durées exactes du run courant sont dans la sortie de la fonction de comparaison ci-dessus ; ce qui ne bouge pas d'un run à l'autre, c'est l'ordre de grandeur de l'écart. Et pendant ce temps le backtracking n'a jamais dépassé 295 appels récursifs.


La leçon n'est pas « le recuit est mauvais » mais **le recuit est le mauvais moteur pour ce problème**. Le Sudoku est un problème à contraintes dures, fortement structuré, où l'inférence logique (propagation, arc-consistance) élimine des continents entiers de l'espace de recherche avant même de compter un pas. Le recuit, lui, n'exploite aucune structure : il perturbe, mesure, accepte ou refuse — adapté aux paysages continus ou aux problèmes sans inférence exploitable, désarmé ici. C'est le critère Prong B en acte : choisir le moteur SOTA *approprié* à la structure du problème, et savoir lire dans un benchmark quand il ne l'est pas.


Le contraste puzzle 2 / puzzle 3 est lui-même instructif : **un facteur une dizaine entre deux instances du même problème** pour le recuit, alors que le backtracking, lui, ne fait guère plus que doubler. Une métaheuristique stochastique a une variance d'exécution énorme selon le paysage local : une grille dont les minimums locaux sont profonds la retient des dizaines de milliers d'itérations. L'exercice 1 (reheat — réchauffage) attaque exactement ce défaut : ré-augmenter la température quand la recherche stagne pour sortir du piège. Aucune de ces rustines ne changera l'ordre de grandeur : sur un problème où l'inférence logique est disponible, le bon moteur reste la recherche avec propagation.

## 10. Exercices

### Exercice 1 : Rechauffement (Reheating)

Le recuit simule peut rester bloque dans des optima locaux. Le **rechauffement** consiste a remonter la temperature pour relancer l'exploration.

**Objectif** : Etendre `Solver` pour ajouter un mécanisme de rechauffement.

**Indices** :
1. Suivez l'energie a chaque palier de temperature
2. Si l'energie ne diminue pas pendant `stagnation_threshold` paliers consecutifs, remontez la temperature
3. Le facteur `reheat_factor` (un facteur inferieur a 1) determine a quel niveau remonter : `T = T0 * reheat_factor`
4. Après rechauffement, continuez le refroidissement normal

**Verification** : Testez sur un puzzle du corpus `Sudoku_top95.txt`, plus difficile qu'Easy51 (`load_puzzles(&puzzles_dir.join("Sudoku_top95.txt"), Some(1))`), avec et sans rechauffement. Le rechauffement devrait augmenter le taux de succes.

In [17]:
// ReheatSolver (stub)
// SplitMix64 — RNG std simple, aucune crate.
struct Rng { state: u64 }
impl Rng {
    fn new(seed: u64) -> Rng { Rng { state: seed } }
    fn next_u64(&mut self) -> u64 {
        self.state = self.state.wrapping_add(0x9e37_79b9_7f4a_7c15);
        let mut z = self.state;
        z = (z ^ (z >> 30)).wrapping_mul(0xbf58_476d_1ce4_e5b9);
        z = (z ^ (z >> 27)).wrapping_mul(0x94d0_49bb_1331_11eb);
        z ^ (z >> 31)
    }
    fn randbelow(&mut self, n: u64) -> u64 { self.next_u64() % n }
    fn random(&mut self) -> f64 { (self.next_u64() >> 11) as f64 / (1u64 << 53) as f64 }
    fn shuffle<T>(&mut self, x: &mut [T]) {
        for i in (1..x.len()).rev() {
            let j = self.randbelow(i as u64 + 1) as usize;
            x.swap(i, j);
        }
    }
    fn sample2(&mut self, population: &[usize]) -> (usize, usize) {
        let n = population.len();
        let mut pool: Vec<usize> = population.to_vec();
        let mut result = [0usize; 2];
        for i in 0..2 {
            let j = self.randbelow((n - i) as u64) as usize;
            result[i] = pool[j];
            pool[j] = pool[n - i - 1];
        }
        (result[0], result[1])
    }
}

#[derive(Clone)]
struct Grid { cells: [[u8; 9]; 9] }

impl Grid {
    fn from_string(s: &str) -> Grid {
        let s = s.replace('.', "0").replace(' ', "").replace('\n', "");
        assert!(s.len() == 81, "La chaine doit avoir 81 caracteres");
        let mut cells = [[0u8; 9]; 9];
        for (i, ch) in s.chars().enumerate() {
            cells[i / 9][i % 9] = ch.to_digit(10).unwrap() as u8;
        }
        Grid { cells }
    }
}

fn compute_energy(grid: &[[u8; 9]; 9]) -> i32 {
    let mut energy = 0i32;
    for col in 0..9 {
        let mut counts = [0i32; 10];
        for r in 0..9 {
            let v = grid[r][col];
            if v > 0 { counts[v as usize] += 1; }
        }
        for v in 1..10 { energy += (counts[v] - 1).max(0); }
    }
    for br in 0..3 {
        for bc in 0..3 {
            let mut counts = [0i32; 10];
            for r in 0..3 {
                for c in 0..3 {
                    let v = grid[br * 3 + r][bc * 3 + c];
                    if v > 0 { counts[v as usize] += 1; }
                }
            }
            for v in 1..10 { energy += (counts[v] - 1).max(0); }
        }
    }
    energy
}

fn initialize_grid(puzzle: &Grid, rng: &mut Rng) -> (Grid, [[bool; 9]; 9]) {
    let mut grid = puzzle.clone();
    let mut is_fixed = [[false; 9]; 9];
    for r in 0..9 {
        for c in 0..9 {
            is_fixed[r][c] = puzzle.cells[r][c] > 0;
        }
    }
    for row in 0..9 {
        let mut fixed = [false; 10];
        for c in 0..9 {
            let v = grid.cells[row][c];
            if v > 0 { fixed[v as usize] = true; }
        }
        let mut missing: Vec<u8> = (1..=9).filter(|&v| !fixed[v as usize]).collect();
        rng.shuffle(&mut missing);
        let mut i = 0;
        for c in 0..9 {
            if grid.cells[row][c] == 0 {
                grid.cells[row][c] = missing[i];
                i += 1;
            }
        }
    }
    (grid, is_fixed)
}

struct ReheatSolver {
    t0: f64,
    alpha: f64,
    iterations_per_temp: usize,
    tmin: f64,
    stagnation_threshold: usize,
    reheat_factor: f64,
}

impl ReheatSolver {
    fn new() -> ReheatSolver {
        ReheatSolver {
            t0: 1.0, alpha: 0.999,
            iterations_per_temp: 100, tmin: 0.001,
            // Exercice: Initialiser les parametres de rechauffement
            stagnation_threshold: 50, // self.stagnation_threshold = stagnation_threshold
            reheat_factor: 0.5,      // self.reheat_factor = reheat_factor
        }
    }

    fn solve(&self, _puzzle: &Grid, _max_restarts: usize) -> (Grid, bool) {
        // Exercice: Copier le code de Solver.solve() et ajouter
        // la detection de stagnation et le rechauffement :
        //   stagnation_count = 0
        //   energy_before_palier = current_energy
        //   ... (boucle interne) ...
        //   if current_energy >= energy_before_palier:
        //       stagnation_count += 1
        //   else:
        //       stagnation_count = 0
        //   if stagnation_count >= self.stagnation_threshold:
        //       T = self.T0 * self.reheat_factor   // Rechauffement
        //       stagnation_count = 0
        //   else:
        //       T *= self.alpha
        (Grid { cells: [[0u8; 9]; 9] }, false)
    }
}

// Test de votre implementation
let here = std::env::current_dir().unwrap();
let mut candidates = vec![
    here.clone(),
    here.join("MyIA.AI.Notebooks").join("Sudoku"),
];
if let Some(parent) = here.parent() {
    candidates.push(parent.to_path_buf().join("Sudoku"));
}
let mut puzzles_dir = std::path::PathBuf::from("Puzzles");
for candidate in &candidates {
    if candidate.join("Puzzles").is_dir() {
        puzzles_dir = candidate.join("Puzzles");
        break;
    }
}
let content = std::fs::read_to_string(puzzles_dir.join("Sudoku_Easy51.txt")).unwrap();
let first_puzzle = content.lines().find(|l| l.trim().len() >= 81).unwrap().trim()[..81].to_string();
let puzzle = Grid::from_string(&first_puzzle);
// let solver = ReheatSolver::new();
// let (result, solved) = solver.solve(&puzzle, 5);
// println!("Resolu: {}", solved);
println!("TODO: Implementez SimulatedAnnealingWithReheatSolver");

TODO: Implementez SimulatedAnnealingWithReheatSolver


### Exercice 2 : Paramètres adaptatifs

Le taux de refroidissement optimal depend du puzzle. Un **contrôle adaptatif** peut ajuster `alpha` dynamiquement selon le comportement du solveur.

**Objectif** : Adapter `alpha` en fonction du taux d'acceptation des mouvements.

**Indices** :
1. A chaque palier, calculez le taux d'acceptation : `accepted_moves / total_moves`
2. Si le taux est eleve (> 0.8), refroidir plus vite : `alpha = min(0.9999, alpha * 1.0001)`
3. Si le taux est faible (< 0.2), refroidir plus lentement : `alpha = max(0.95, alpha * 0.9999)`
4. `alpha` reste dans [0.95, 0.9999]

**Verification** : Comparez `alpha` constant vs adaptatif sur 10 puzzles du corpus `Sudoku_top95.txt` (`load_puzzles(&puzzles_dir.join("Sudoku_top95.txt"), Some(10))`). Mesurez le taux de succes moyen.

In [18]:
// AdaptiveSolver (stub)
// SplitMix64 — RNG std simple, aucune crate.
struct Rng { state: u64 }
impl Rng {
    fn new(seed: u64) -> Rng { Rng { state: seed } }
    fn next_u64(&mut self) -> u64 {
        self.state = self.state.wrapping_add(0x9e37_79b9_7f4a_7c15);
        let mut z = self.state;
        z = (z ^ (z >> 30)).wrapping_mul(0xbf58_476d_1ce4_e5b9);
        z = (z ^ (z >> 27)).wrapping_mul(0x94d0_49bb_1331_11eb);
        z ^ (z >> 31)
    }
    fn randbelow(&mut self, n: u64) -> u64 { self.next_u64() % n }
    fn random(&mut self) -> f64 { (self.next_u64() >> 11) as f64 / (1u64 << 53) as f64 }
    fn shuffle<T>(&mut self, x: &mut [T]) {
        for i in (1..x.len()).rev() {
            let j = self.randbelow(i as u64 + 1) as usize;
            x.swap(i, j);
        }
    }
    fn sample2(&mut self, population: &[usize]) -> (usize, usize) {
        let n = population.len();
        let mut pool: Vec<usize> = population.to_vec();
        let mut result = [0usize; 2];
        for i in 0..2 {
            let j = self.randbelow((n - i) as u64) as usize;
            result[i] = pool[j];
            pool[j] = pool[n - i - 1];
        }
        (result[0], result[1])
    }
}

#[derive(Clone)]
struct Grid { cells: [[u8; 9]; 9] }

impl Grid {
    fn from_string(s: &str) -> Grid {
        let s = s.replace('.', "0").replace(' ', "").replace('\n', "");
        assert!(s.len() == 81, "La chaine doit avoir 81 caracteres");
        let mut cells = [[0u8; 9]; 9];
        for (i, ch) in s.chars().enumerate() {
            cells[i / 9][i % 9] = ch.to_digit(10).unwrap() as u8;
        }
        Grid { cells }
    }
}

fn compute_energy(grid: &[[u8; 9]; 9]) -> i32 {
    let mut energy = 0i32;
    for col in 0..9 {
        let mut counts = [0i32; 10];
        for r in 0..9 {
            let v = grid[r][col];
            if v > 0 { counts[v as usize] += 1; }
        }
        for v in 1..10 { energy += (counts[v] - 1).max(0); }
    }
    for br in 0..3 {
        for bc in 0..3 {
            let mut counts = [0i32; 10];
            for r in 0..3 {
                for c in 0..3 {
                    let v = grid[br * 3 + r][bc * 3 + c];
                    if v > 0 { counts[v as usize] += 1; }
                }
            }
            for v in 1..10 { energy += (counts[v] - 1).max(0); }
        }
    }
    energy
}

fn initialize_grid(puzzle: &Grid, rng: &mut Rng) -> (Grid, [[bool; 9]; 9]) {
    let mut grid = puzzle.clone();
    let mut is_fixed = [[false; 9]; 9];
    for r in 0..9 {
        for c in 0..9 {
            is_fixed[r][c] = puzzle.cells[r][c] > 0;
        }
    }
    for row in 0..9 {
        let mut fixed = [false; 10];
        for c in 0..9 {
            let v = grid.cells[row][c];
            if v > 0 { fixed[v as usize] = true; }
        }
        let mut missing: Vec<u8> = (1..=9).filter(|&v| !fixed[v as usize]).collect();
        rng.shuffle(&mut missing);
        let mut i = 0;
        for c in 0..9 {
            if grid.cells[row][c] == 0 {
                grid.cells[row][c] = missing[i];
                i += 1;
            }
        }
    }
    (grid, is_fixed)
}

fn generate_neighbor(grid: &mut Grid, is_fixed: &[[bool; 9]; 9], rng: &mut Rng) -> (usize, usize, usize) {
    let row;
    loop {
        let r = rng.randbelow(9) as usize;
        let free = (0..9).filter(|&c| !is_fixed[r][c]).count();
        if free >= 2 { row = r; break; }
    }
    let free_cols: Vec<usize> = (0..9).filter(|&c| !is_fixed[row][c]).collect();
    let (col1, col2) = rng.sample2(&free_cols);
    grid.cells[row].swap(col1, col2);
    (row, col1, col2)
}

fn undo_swap(grid: &mut Grid, row: usize, col1: usize, col2: usize) {
    grid.cells[row].swap(col1, col2);
}

struct AdaptiveSolver {
    t0: f64,
    alpha: f64,
    iterations_per_temp: usize,
    tmin: f64,
}

impl AdaptiveSolver {
    fn new() -> AdaptiveSolver {
        AdaptiveSolver { t0: 1.0, alpha: 0.999, iterations_per_temp: 100, tmin: 0.001 }
    }

    fn solve(&self, _puzzle: &Grid, _max_restarts: usize) -> (Grid, bool) {
        let mut rng = Rng::new(42);
        let mut _best_grid: Option<Grid> = None;
        let mut _best_energy = f64::INFINITY;

        for _restart in 0.._max_restarts {
            let (_grid, _is_fixed) = initialize_grid(_puzzle, &mut rng);
            let _current_energy = compute_energy(&_grid.cells);
            let mut t = self.t0;
            let mut alpha = self.alpha; // alpha variable maintenant

            while t > self.tmin && _current_energy > 0 {
                let _accepted = 0;
                let mut total = 0;

                for _ in 0..self.iterations_per_temp {
                    // Exercice: Generer voisin et appliquer critere de Metropolis
                    // let (row, col1, col2) = generate_neighbor(&mut grid, &is_fixed, &mut rng);
                    // let new_energy = compute_energy(&grid.cells);
                    // let delta_E = new_energy - current_energy;
                    // if delta_E <= 0 || rng.random() < (-delta_E as f64 / t).exp():
                    //     accepted += 1;
                    //     current_energy = new_energy;
                    // else:
                    //     undo_swap(&mut grid, row, col1, col2);
                    total += 1;
                }

                // Exercice: Adapter alpha selon le taux d'acceptation
                // let acceptance_rate = _accepted as f64 / total as f64;
                // if acceptance_rate > 0.8 {
                //     alpha = (0.9999f64).min(alpha * 1.0001);
                // } else if acceptance_rate < 0.2 {
                //     alpha = (0.95f64).max(alpha * 0.9999);
                // }

                t *= alpha;
            }
        }

        (_best_grid.unwrap_or(Grid { cells: [[0u8; 9]; 9] }), _best_energy == 0.0)
    }
}

// Test de votre implementation
let here = std::env::current_dir().unwrap();
let mut candidates = vec![
    here.clone(),
    here.join("MyIA.AI.Notebooks").join("Sudoku"),
];
if let Some(parent) = here.parent() {
    candidates.push(parent.to_path_buf().join("Sudoku"));
}
let mut puzzles_dir = std::path::PathBuf::from("Puzzles");
for candidate in &candidates {
    if candidate.join("Puzzles").is_dir() {
        puzzles_dir = candidate.join("Puzzles");
        break;
    }
}
let content = std::fs::read_to_string(puzzles_dir.join("Sudoku_Easy51.txt")).unwrap();
let first_puzzle = content.lines().find(|l| l.trim().len() >= 81).unwrap().trim()[..81].to_string();
let puzzle = Grid::from_string(&first_puzzle);
let solver = AdaptiveSolver::new();
// let (result, solved) = solver.solve(&puzzle, 5);
// println!("Resolu: {}", solved);
println!("TODO: Implementez AdaptiveSimulatedAnnealingSolver");

TODO: Implementez AdaptiveSimulatedAnnealingSolver


### Exercice 3 : Voisinage etendu

Le voisinage par echange intra-ligne est local. Des mouvements plus globaux peuvent aider a echapper aux optima locaux.

**Objectif** : Implementer un voisinage etendu avec des echanges inter-lignes.

**Indices** :
1. **Echange intra-ligne** (standard) : echanger deux cases non fixes dans la même ligne
2. **Echange inter-lignes** (etendu) : echanger une case non fixe en ligne `row1` avec une case non fixe en ligne `row1 + 1`
3. Avec probabilite `use_extended`, utiliser l'echange inter-lignes; sinon l'echange standard
4. Retourner `(move_type, params)` pour pouvoir annuler le mouvement

**Verification** : Testez différentes valeurs de `use_extended` (0.1, 0.2, 0.5) sur des puzzles du corpus `Sudoku_top95.txt`.

In [19]:
// generate_extended_neighbor (stub)
// SplitMix64 — RNG std simple, aucune crate.
struct Rng { state: u64 }
impl Rng {
    fn new(seed: u64) -> Rng { Rng { state: seed } }
    fn next_u64(&mut self) -> u64 {
        self.state = self.state.wrapping_add(0x9e37_79b9_7f4a_7c15);
        let mut z = self.state;
        z = (z ^ (z >> 30)).wrapping_mul(0xbf58_476d_1ce4_e5b9);
        z = (z ^ (z >> 27)).wrapping_mul(0x94d0_49bb_1331_11eb);
        z ^ (z >> 31)
    }
    fn randbelow(&mut self, n: u64) -> u64 { self.next_u64() % n }
    fn random(&mut self) -> f64 { (self.next_u64() >> 11) as f64 / (1u64 << 53) as f64 }
    fn shuffle<T>(&mut self, x: &mut [T]) {
        for i in (1..x.len()).rev() {
            let j = self.randbelow(i as u64 + 1) as usize;
            x.swap(i, j);
        }
    }
    fn sample2(&mut self, population: &[usize]) -> (usize, usize) {
        let n = population.len();
        let mut pool: Vec<usize> = population.to_vec();
        let mut result = [0usize; 2];
        for i in 0..2 {
            let j = self.randbelow((n - i) as u64) as usize;
            result[i] = pool[j];
            pool[j] = pool[n - i - 1];
        }
        (result[0], result[1])
    }
}

#[derive(Clone)]
struct Grid { cells: [[u8; 9]; 9] }

impl Grid {
    fn from_string(s: &str) -> Grid {
        let s = s.replace('.', "0").replace(' ', "").replace('\n', "");
        assert!(s.len() == 81, "La chaine doit avoir 81 caracteres");
        let mut cells = [[0u8; 9]; 9];
        for (i, ch) in s.chars().enumerate() {
            cells[i / 9][i % 9] = ch.to_digit(10).unwrap() as u8;
        }
        Grid { cells }
    }
}

fn compute_energy(grid: &[[u8; 9]; 9]) -> i32 {
    let mut energy = 0i32;
    for col in 0..9 {
        let mut counts = [0i32; 10];
        for r in 0..9 {
            let v = grid[r][col];
            if v > 0 { counts[v as usize] += 1; }
        }
        for v in 1..10 { energy += (counts[v] - 1).max(0); }
    }
    for br in 0..3 {
        for bc in 0..3 {
            let mut counts = [0i32; 10];
            for r in 0..3 {
                for c in 0..3 {
                    let v = grid[br * 3 + r][bc * 3 + c];
                    if v > 0 { counts[v as usize] += 1; }
                }
            }
            for v in 1..10 { energy += (counts[v] - 1).max(0); }
        }
    }
    energy
}

fn initialize_grid(puzzle: &Grid, rng: &mut Rng) -> (Grid, [[bool; 9]; 9]) {
    let mut grid = puzzle.clone();
    let mut is_fixed = [[false; 9]; 9];
    for r in 0..9 {
        for c in 0..9 {
            is_fixed[r][c] = puzzle.cells[r][c] > 0;
        }
    }
    for row in 0..9 {
        let mut fixed = [false; 10];
        for c in 0..9 {
            let v = grid.cells[row][c];
            if v > 0 { fixed[v as usize] = true; }
        }
        let mut missing: Vec<u8> = (1..=9).filter(|&v| !fixed[v as usize]).collect();
        rng.shuffle(&mut missing);
        let mut i = 0;
        for c in 0..9 {
            if grid.cells[row][c] == 0 {
                grid.cells[row][c] = missing[i];
                i += 1;
            }
        }
    }
    (grid, is_fixed)
}

fn generate_neighbor(grid: &mut Grid, is_fixed: &[[bool; 9]; 9], rng: &mut Rng) -> (usize, usize, usize) {
    let row;
    loop {
        let r = rng.randbelow(9) as usize;
        let free = (0..9).filter(|&c| !is_fixed[r][c]).count();
        if free >= 2 { row = r; break; }
    }
    let free_cols: Vec<usize> = (0..9).filter(|&c| !is_fixed[row][c]).collect();
    let (col1, col2) = rng.sample2(&free_cols);
    grid.cells[row].swap(col1, col2);
    (row, col1, col2)
}

fn undo_swap(grid: &mut Grid, row: usize, col1: usize, col2: usize) {
    grid.cells[row].swap(col1, col2);
}

fn generate_extended_neighbor(_grid: &mut Grid, _is_fixed: &[[bool; 9]; 9], _rng: &mut Rng, _use_extended: f64) -> (String, (usize, usize, usize, usize)) {
    // Genere un voisin avec probabilite use_extended d'utiliser un mouvement inter-lignes.
    // Retourne (move_type, params) pour pouvoir annuler le mouvement.
    if _rng.random() < _use_extended {
        // Exercice: Mouvement etendu - echange entre lignes adjacentes
        // Indice 1 : let row1 = rng.randbelow(8) as usize; let row2 = row1 + 1;
        // Indice 2 : Choisir col1 non fixe dans row1, col2 non fixe dans row2
        // Indice 3 : Effectuer l'echange : grid[row1][col1] <-> grid[row2][col2]
        // Indice 4 : Retourner ("inter_line".to_string(), (row1, col1, row2, col2))
        ("inter_line".to_string(), (0, 0, 0, 0))
    } else {
        // Exercice: Mouvement standard - echange intra-ligne
        // Indice : Utiliser generate_neighbor() et retourner ("intra_line".to_string(), (row, col1, col2, 0))
        ("intra_line".to_string(), (0, 0, 0, 0))
    }
}

fn undo_extended_move(_grid: &mut Grid, _move_type: &str, _params: (usize, usize, usize, usize)) {
    // Annule un mouvement etendu.
    // Exercice: Implementer selon move_type
    // if move_type == "intra_line":
    //     let (row, col1, col2, _) = params;
    //     undo_swap(grid, row, col1, col2);
    // else if move_type == "inter_line":
    //     let (row1, col1, row2, col2) = params;
    //     grid.cells[row1][col1] = grid.cells[row2][col2];
    //     grid.cells[row2][col2] = saved;
}

// Test de votre implementation
println!("TODO: Implementez generate_extended_neighbor et undo_extended_move");
// let (grid_test, is_fixed_test) = initialize_grid(&Grid::from_string(&first_puzzle), &mut Rng::new(42));
// let (move_type, params) = generate_extended_neighbor(&mut grid_test, &is_fixed_test, &mut Rng::new(42), 1.0);
// println!("Mouvement: {}, params: {:?}", move_type, params);

TODO: Implementez generate_extended_neighbor et undo_extended_move


## Synthèse

### Concepts clés

| Concept | Description |
|---------|-------------|
| **Recuit simule** | Métaheuristique d'optimisation inspiree de la metallurgie |
| **Fonction d'energie** | Nombre de doublons colonnes + blocs |
| **Voisinage** | Echange de deux cases non fixes dans une même ligne |
| **Acceptation de Metropolis** | Accepter les degradations avec probabilite $e^{-\Delta E / T}$ |
| **Refroidissement** | Reduction progressive de $T$ (programme exponentiel) |

### Forces et limites

| Aspect | Avantage | Inconvenient |
|--------|----------|-------------|
| Garantie | Aucune | Ne trouve pas toujours la solution |
| Performance | Variable selon paramètres | Plus lent que les solveurs CSP |
| Pedagogie | Elegant pour comprendre l'optimisation | Pas adapte aux CSP stricts |

### Quand l'utiliser

- Puzzles moyens (pas trop difficiles)
- Quand on veut comprendre les métaheuristiques
- Pour explorer l'espace de solutions

### Alternatives recommandees

- **Backtracking** : [Sudoku-01-Backtracking-Python](Sudoku-01-Backtracking-Python.ipynb)
- **OR-Tools CP-SAT** : [Sudoku-10-ORTools-Python](Sudoku-10-ORTools-Python.ipynb)
- **Z3 SMT** : [Sudoku-12-Z3-Python](Sudoku-12-Z3-Python.ipynb)

***

**Navigation** : [<< Sudoku-03-Genetic](Sudoku-03-Genetic-Python.ipynb) | [Index](README.md) | [Sudoku-05-PSO >>](Sudoku-05-PSO-Python.ipynb) | [jumeau Python](Sudoku-04-SimulatedAnnealing-Python.ipynb)

## Synthèse

Ce notebook a applique le **recuit simule (Simulated Annealing)** a la resolution de Sudoku.

### Formulation SA pour le Sudoku

| Élément | Choix |
|---------|-------|
| **État** | Grille 9x9 avec permutations par ligne |
| **Energie** | Doublons colonnes + blocs 3x3 (lignes valides par construction) |
| **Voisinage** | Echange de 2 cases non-fixees dans la même ligne |
| **Refroidissement** | T *= alpha (0.999), T0=1.0, Tmin=0.001 |

### Résultats du benchmark

| Méthode | Puzzles | Temps observe |
|---------|---------|---------------|
| **SA manuel** | 3/3 resolus | mesurable en direct (Section 8, `benchmark_sa` ; croit avec le nombre de vides) |
| **Backtracking** | 3/3 resolus | 49-295 appels (déterministe ; durée en direct Section 9, `Backtracking`) |
| **simanneal (50000 steps)** | 1/1 resolu | mesurable en direct (Section 6, test simanneal) |

### Lecture comparee

Le backtracking resout ces puzzles faciles en quelques millisecondes ; le recuit simule y parvient aussi mais est **considerablement plus lent** (la durée exacte sur les cases vides se mesure en direct via la fonction `benchmark_sa`, Section 8 — SA stochastique). La valeur du recuit simule n'est pas dans la vitesse sur ces instances faciles, mais dans sa capacité a **explorer un paysage d'energie** (acceptation de mouvements degradants a haute temperature, exploitation a basse temperature) -- une approche transferable a des problèmes d'optimisation ou aucun solveur déterministe n'est disponible.


Le recuit simule offre une approche **non-déterministe** sans garantie de completude, adaptee aux problèmes d'optimisation combinatoire. Pour le Sudoku, les solveurs CSP (OR-Tools CP-SAT, Z3 SMT) restent plus fiables sur les instances difficiles.

**Suite** : [Sudoku-05 - PSO](Sudoku-05-PSO-Python.ipynb) | [Sudoku-10 - OR-Tools](Sudoku-10-ORTools-Python.ipynb) | [Retour au sommaire](README.md)

## 11. Comparaison quantitative SA vs Backtracking (Prong B #3801)

Le benchmark précédent (Section 8, fonction `benchmark_sa`) a mesure le recuit simule (SA) sur **trois puzzles faciles** (de 36 a 51 cases vides) avec un temps culminant mesurable en direct dans sa sortie sur le puzzle 3 (51 cases a remplir). Pour positionner SA face au solveur exact (Backtracking) sur les mêmes instances, voici une **comparaison quantitative** avec les chiffres reels publies dans les fonctions voisines (Section 8 `benchmark_sa` pour SA, Section 9 `Backtracking` pour BT).

### Tableau comparatif (chiffres verifies, sources citees)

| Solveur | Puzzle 1 (36 vides) | Puzzle 2 (49 vides) | Puzzle 3 (51 vides) | Garantie exactitude |
|---------|---------------------|---------------------|---------------------|---------------------|
| **Backtracking simple** (`Backtracking`, Section 9) | 49 appels, quelques ms | 201 appels, quelques ms | 295 appels, quelques dizaines de ms | Oui (exhaustif) |
| **Recuit Simule** (`Solver`, Section 9) | OK, dixièmes de seconde | OK, dizaines de secondes | OK, plusieurs minutes | Non (heuristique) |
| **Ratio SA / BT** (Section 9) | ~deux ordres | ~trois ordres | ~quatre ordres | -- |

### Observations (Prong B illustre)

1. **Ecart de plusieurs ordres de grandeur** entre SA et Backtracking sur les mêmes instances. Le ratio explose avec la difficulte (cases vides) : de deux ordres de grandeur (36 vides) a quatre ordres (51 vides) sur le run mesure -- re-mesurez-le en direct via les fonctions `benchmark_sa` (Section 8) et `Backtracking` (Section 9).
2. **Garantie d'exactitude** : Backtracking est **complet** (trouve une solution si elle existe), SA est **non garanti** (les 3 OK ci-dessus dependent du seed et du time budget).
3. **Coût runtime** : SA devient lent (plusieurs minutes) sur le puzzle 3 (51 vides), Backtracking reste en millisecondes. Pour la production, Backtracking (ou DLX) est systematiquement preferable.
4. **Valeur pédagogique du SA** : illustre une **famille de méthodes** (recherche locale + stochasticite) complémentaire des solveurs exacts. Le SA brille sur des problèmes ou l'espace de recherche est trop grand pour l'exhaustif et ou une solution "proche de l'optimal" suffit (TSP, planning, scheduling). Pour le Sudoku, ce n'est **pas** le cas : backtracking + danse-links (DLX) dominent.

> **Note methodologique** : les durées absolues sont mesurees en direct par les fonctions benchmark (`benchmark_sa` en Section 8 pour SA, `Backtracking` en Section 9 pour BT ; outputs reels, 0 erreur) -- re-executez-les pour les valeurs courantes. SA etant stochastique, deux runs différent : deux passes consecutives des Sections 8 et 9 donnent des durées différentes, et c'est la variance attendue du recuit, pas une erreur ; les valeurs du run courant sont dans leurs sorties respectives. Les constantes `T_SA_MS` / `T_BT_MS` de l'exercice (Section 11) sont **derivees** de la mesure de la Section 9 : elles suivent ce run au lieu d'etre recopiees, et ne peuvent donc plus citer des durées absentes des sorties. Seuls les compteurs d'appels Backtracking (49/201/295) sont déterministes. Le recuit simule est execute avec `T0=1.0, alpha=0.999, iterations_per_temp=100, Tmin=0.001, max_restarts=10` (appel de `benchmark_sa`, Section 8).

## Exercice : Estimer le ratio SA / Backtracking (Prong B reflexif)

### Contexte

Le tableau ci-dessus compare le recuit simule (heuristique stochastique) au Backtracking (solveur exact) sur 3 puzzles Easy du même fichier. Pour faire toucher du doigt le **ratio de coût** et sa **non-linearite avec la difficulte**, on veut le chiffrer — a partir des **durées mesurees en direct** par la fonction `Backtracking` (Section 9), qui chronometre les deux moteurs sur les mêmes instances. La fonction suivante les expose en constantes (`T_SA_MS`, `T_BT_MS`) **derivees de ce run** : elles ne sont plus recopiees a la main.

### Enonce

A partir des constantes `T_SA_MS` / `T_BT_MS`, derivees des **durées mesurees en direct** par la fonction `Backtracking` (Section 9) :

1. Calculer le ratio `temps_SA / temps_BT` pour chacun des 3 puzzles.
2. Observer la **non-linearite** : le ratio explose-t-il lineairement avec le nombre de cases vides, ou exponentiellement ?
3. Conclure sur la **position du SA dans la hiérarchie** : efficace / equivalentes / nettement inferieures / sans commune mesure, sur le cas Sudoku.

### Indication

- `ratio_puzzle_X = t_sa_X_ms / t_bt_X_ms`
- Cases vides : 36 -> 49 -> 51 (Puzzles 1 -> 2 -> 3)
- Si `ratio_puzzle_3 / ratio_puzzle_1 > 10`, on a une **croissance super-lineaire** -> méthode inadaptee au scaling
- Suggestion : `conclude_prong_b()` retourne une catégorie parmi `"lineaire"`, `"polynomial"`, `"exponentiel"`, `"autre"`

### Solution (a completer par l'etudiant)

```
ratio_puzzle_1 = None  // TODO etudiant
ratio_puzzle_2 = None  // TODO etudiant
ratio_puzzle_3 = None  // TODO etudiant
conclusion     = None  // TODO etudiant
```

In [20]:
// Exercice : Estimer le ratio SA / Backtracking (Prong B reflexif)
// SplitMix64 — RNG std simple, aucune crate.
struct Rng { state: u64 }
impl Rng {
    fn new(seed: u64) -> Rng { Rng { state: seed } }
    fn next_u64(&mut self) -> u64 {
        self.state = self.state.wrapping_add(0x9e37_79b9_7f4a_7c15);
        let mut z = self.state;
        z = (z ^ (z >> 30)).wrapping_mul(0xbf58_476d_1ce4_e5b9);
        z = (z ^ (z >> 27)).wrapping_mul(0x94d0_49bb_1331_11eb);
        z ^ (z >> 31)
    }
    fn randbelow(&mut self, n: u64) -> u64 { self.next_u64() % n }
    fn random(&mut self) -> f64 { (self.next_u64() >> 11) as f64 / (1u64 << 53) as f64 }
    fn shuffle<T>(&mut self, x: &mut [T]) {
        for i in (1..x.len()).rev() {
            let j = self.randbelow(i as u64 + 1) as usize;
            x.swap(i, j);
        }
    }
    fn sample2(&mut self, population: &[usize]) -> (usize, usize) {
        let n = population.len();
        let mut pool: Vec<usize> = population.to_vec();
        let mut result = [0usize; 2];
        for i in 0..2 {
            let j = self.randbelow((n - i) as u64) as usize;
            result[i] = pool[j];
            pool[j] = pool[n - i - 1];
        }
        (result[0], result[1])
    }
}

#[derive(Clone)]
struct Grid { cells: [[u8; 9]; 9] }

impl Grid {
    fn from_string(s: &str) -> Grid {
        let s = s.replace('.', "0").replace(' ', "").replace('\n', "");
        assert!(s.len() == 81, "La chaine doit avoir 81 caracteres");
        let mut cells = [[0u8; 9]; 9];
        for (i, ch) in s.chars().enumerate() {
            cells[i / 9][i % 9] = ch.to_digit(10).unwrap() as u8;
        }
        Grid { cells }
    }
}

fn compute_energy(grid: &[[u8; 9]; 9]) -> i32 {
    let mut energy = 0i32;
    for col in 0..9 {
        let mut counts = [0i32; 10];
        for r in 0..9 {
            let v = grid[r][col];
            if v > 0 { counts[v as usize] += 1; }
        }
        for v in 1..10 { energy += (counts[v] - 1).max(0); }
    }
    for br in 0..3 {
        for bc in 0..3 {
            let mut counts = [0i32; 10];
            for r in 0..3 {
                for c in 0..3 {
                    let v = grid[br * 3 + r][bc * 3 + c];
                    if v > 0 { counts[v as usize] += 1; }
                }
            }
            for v in 1..10 { energy += (counts[v] - 1).max(0); }
        }
    }
    energy
}

fn initialize_grid(puzzle: &Grid, rng: &mut Rng) -> (Grid, [[bool; 9]; 9]) {
    let mut grid = puzzle.clone();
    let mut is_fixed = [[false; 9]; 9];
    for r in 0..9 {
        for c in 0..9 {
            is_fixed[r][c] = puzzle.cells[r][c] > 0;
        }
    }
    for row in 0..9 {
        let mut fixed = [false; 10];
        for c in 0..9 {
            let v = grid.cells[row][c];
            if v > 0 { fixed[v as usize] = true; }
        }
        let mut missing: Vec<u8> = (1..=9).filter(|&v| !fixed[v as usize]).collect();
        rng.shuffle(&mut missing);
        let mut i = 0;
        for c in 0..9 {
            if grid.cells[row][c] == 0 {
                grid.cells[row][c] = missing[i];
                i += 1;
            }
        }
    }
    (grid, is_fixed)
}

fn generate_neighbor(grid: &mut Grid, is_fixed: &[[bool; 9]; 9], rng: &mut Rng) -> (usize, usize, usize) {
    let row;
    loop {
        let r = rng.randbelow(9) as usize;
        let free = (0..9).filter(|&c| !is_fixed[r][c]).count();
        if free >= 2 { row = r; break; }
    }
    let free_cols: Vec<usize> = (0..9).filter(|&c| !is_fixed[row][c]).collect();
    let (col1, col2) = rng.sample2(&free_cols);
    grid.cells[row].swap(col1, col2);
    (row, col1, col2)
}

fn undo_swap(grid: &mut Grid, row: usize, col1: usize, col2: usize) {
    grid.cells[row].swap(col1, col2);
}

struct Solver {
    t0: f64,
    alpha: f64,
    iterations_per_temp: usize,
    tmin: f64,
}

impl Solver {
    fn default() -> Solver {
        Solver { t0: 1.0, alpha: 0.999, iterations_per_temp: 100, tmin: 0.001 }
    }

    fn solve(&self, puzzle: &Grid, max_restarts: usize) -> (Grid, bool) {
        let mut rng = Rng::new(42);
        let mut best_grid: Option<Grid> = None;
        let mut best_energy = f64::INFINITY;

        for _restart in 0..max_restarts {
            let (mut grid, is_fixed) = initialize_grid(puzzle, &mut rng);
            let mut current_energy = compute_energy(&grid.cells);

            if (current_energy as f64) < best_energy {
                best_energy = current_energy as f64;
                best_grid = Some(grid.clone());
            }
            if current_energy == 0 {
                return (best_grid.unwrap(), true);
            }

            let mut t = self.t0;
            while t > self.tmin && current_energy > 0 {
                for _ in 0..self.iterations_per_temp {
                    let (row, col1, col2) = generate_neighbor(&mut grid, &is_fixed, &mut rng);
                    let new_energy = compute_energy(&grid.cells);
                    let delta_e = new_energy - current_energy;
                    if delta_e <= 0 || rng.random() < (-delta_e as f64 / t).exp() {
                        current_energy = new_energy;
                        if (current_energy as f64) < best_energy {
                            best_energy = current_energy as f64;
                            best_grid = Some(grid.clone());
                        }
                        if current_energy == 0 {
                            return (best_grid.unwrap(), true);
                        }
                    } else {
                        undo_swap(&mut grid, row, col1, col2);
                    }
                }
                t *= self.alpha;
            }

            if best_energy == 0.0 {
                return (best_grid.unwrap(), true);
            }
        }

        (best_grid.unwrap(), best_energy == 0.0)
    }
}

struct Backtracking { call_count: u64 }

impl Backtracking {
    fn new() -> Backtracking { Backtracking { call_count: 0 } }

    fn solve(&mut self, puzzle: &Grid) -> bool {
        let mut grid = puzzle.cells;
        self.call_count = 0;
        self._backtrack(&mut grid)
    }

    fn _backtrack(&mut self, grid: &mut [[u8; 9]; 9]) -> bool {
        self.call_count += 1;
        for r in 0..9 {
            for c in 0..9 {
                if grid[r][c] == 0 {
                    for num in 1..=9u8 {
                        if self._is_valid(grid, r, c, num) {
                            grid[r][c] = num;
                            if self._backtrack(grid) {
                                return true;
                            }
                            grid[r][c] = 0;
                        }
                    }
                    return false;
                }
            }
        }
        true
    }

    fn _is_valid(&self, grid: &[[u8; 9]; 9], row: usize, col: usize, num: u8) -> bool {
        for c in 0..9 {
            if grid[row][c] == num { return false; }
        }
        for r in 0..9 {
            if grid[r][col] == num { return false; }
        }
        let br = 3 * (row / 3);
        let bc = 3 * (col / 3);
        for r in br..br + 3 {
            for c in bc..bc + 3 {
                if grid[r][c] == num { return false; }
            }
        }
        true
    }
}

// Stubs pour l'exercice
fn compute_ratio_sa_bt(_sa_ms: f64, _bt_ms: f64) -> Option<f64> {
    None // TODO etudiant
}

fn conclude_prong_b(_ratios: &[Option<f64>]) -> Option<&str> {
    None // TODO etudiant
}

// Rejoue la comparaison (autonomie evcxr)
let here = std::env::current_dir().unwrap();
let mut candidates = vec![
    here.clone(),
    here.join("MyIA.AI.Notebooks").join("Sudoku"),
];
if let Some(parent) = here.parent() {
    candidates.push(parent.to_path_buf().join("Sudoku"));
}
let mut puzzles_dir = std::path::PathBuf::from("Puzzles");
for candidate in &candidates {
    if candidate.join("Puzzles").is_dir() {
        puzzles_dir = candidate.join("Puzzles");
        break;
    }
}
let content = std::fs::read_to_string(puzzles_dir.join("Sudoku_Easy51.txt")).unwrap();
let easy_puzzles: Vec<String> = content.lines()
    .filter(|l| l.trim().len() >= 81)
    .take(5)
    .map(|l| l.trim()[..81].to_string())
    .collect();

// Mesurer les durees
let mut t_bt_ms: Vec<f64> = Vec::new();
let mut t_sa_ms: Vec<f64> = Vec::new();

for i in 0..3 {
    let grid = Grid::from_string(&easy_puzzles[i]);

    // Backtracking
    let mut bt = Backtracking::new();
    let start = std::time::Instant::now();
    bt.solve(&grid);
    let t_bt = start.elapsed().as_secs_f64() * 1000.0;
    t_bt_ms.push(t_bt);

    // Recuit simule
    let solver = Solver::default();
    let start = std::time::Instant::now();
    solver.solve(&grid, 5);
    let t_sa = start.elapsed().as_secs_f64() * 1000.0;
    t_sa_ms.push(t_sa);
}

// Vides calcules
let vides: Vec<usize> = easy_puzzles.iter().take(3).map(|p| {
    let g = Grid::from_string(p);
    (0..9).flat_map(|r| (0..9).map(move |c| (r, c))).filter(|&(r, c)| g.cells[r][c] == 0).count()
}).collect();

// Affichage
println!("=== Estimation du ratio SA / Backtracking (Prong B #3801) ===");
println!();
println!("Puzzle      Vides    BT (ms)      SA (ms)    Ratio SA/BT");
println!("--------------------------------------------------------");

for i in 0..3 {
    let bt_val = (t_bt_ms[i] * 10.0).round() / 10.0;
    let sa_val = (t_sa_ms[i] * 10.0).round() / 10.0;
    println!("Puzzle {:<3} {:>6} {:>10.1} {:>12.1} {:>14}", i + 1, vides[i], bt_val, sa_val, "None");
}

println!();
println!("Croissance ratio P3/P1 : N/A (a completer)");
println!("Conclusion Prong B    : None");
println!();
println!(">>> Exercice a completer : implementer compute_ratio_sa_bt() et conclude_prong_b()");

=== Estimation du ratio SA / Backtracking (Prong B #3801) ===


Puzzle      Vides    BT (ms)      SA (ms)    Ratio SA/BT


--------------------------------------------------------


Puzzle 1       36        0.0          0.4           None


Puzzle 2       49        0.0         43.6           None


Puzzle 3       51        0.0        296.7           None


Croissance ratio P3/P1 : N/A (a completer)


Conclusion Prong B    : None


>>> Exercice a completer : implementer compute_ratio_sa_bt() et conclude_prong_b()


### References citees dans la section 11 (chiffres verifies)

- `MyIA.AI.Notebooks/Sudoku/Sudoku-04-SimulatedAnnealing-Python.ipynb` (Section 8, fonction `benchmark_sa` — benchmark SA sur 3 Easy)
- `MyIA.AI.Notebooks/Sudoku/Sudoku-04-SimulatedAnnealing-Python.ipynb` (Section 9, fonction `Backtracking` — comparaison BT vs SA sur 3 Easy)

### Liens transverses (autres solveurs Sudoku pour Prong B)

- `MyIA.AI.Notebooks/Sudoku/Sudoku-01-Backtracking-Python.ipynb` (cell 15 — Backtracking benchmark)
- `MyIA.AI.Notebooks/Sudoku/Sudoku-10-ORTools-Python.ipynb` (cell 19 — OR-Tools CP-SAT benchmark)
- `MyIA.AI.Notebooks/Sudoku/Sudoku-03-Genetic-Python.ipynb` (cell 22 — Genetic Algorithm benchmark)

> **Note pédagogique** : ces ratios (SA/BT, de deux a quatre ordres de grandeur selon la difficulte, sur le run mesure en Section 9) demontrent que **le recuit simule est inadapte au Sudoku en production**, malgre sa richesse théorique. Cf. tableau comparatif dans la Section 7 de Sudoku-03-Genetic-Python (ratio GA/OR-Tools ~700x sur Easy).

## À retenir

Le **recuit simulé** résout le Sudoku comme un problème d'optimisation : on minimise l'**énergie** (= nombre de conflits), en acceptant parfois un écart selon une « température » qui décroît, pour sortir des minima locaux. L'énergie initiale de la grille complétée est **positive** ; l'objectif est **0**.

- Le test avec la fonction `anneal_simanneal` trouve la solution en **quelques millisecondes** (énergie finale **0**, solution valide) ; une variante « manuelle » du même recuit résout le puzzle témoin en **moins d'une seconde**.
- Sur un benchmark de 3 puzzles, les **3/3 sont résolus** en **0,36 s au total, 0,12 s en moyenne** (cellule 33) — le recuit converge, et vite sur ces instances faciles.
- Face au backtracking, le recuit reste **bien plus lent** : sur le puzzle 3 (51 cases vides), le backtracking fait **295 appels en 0,0 ms affichés** là où le recuit met **296 ms** (cellule 36) — l'optimisation heuristique « paie » l'exploration que l'exactitude évite.
- **Comparaison avec le jumeau Python**, pour situer ces durées : ses sorties enregistrées donnent **16,12 s** pour le premier test, **151,46 s** de moyenne au benchmark et **426 386 ms** pour le recuit du puzzle 3 — les valeurs ci-dessus, relevées dans les sorties Rust de ce carnet, sont de l'ordre de **500 à 1 400 fois** plus courtes (mêmes grilles, même machine).

## Vérifiez votre compréhension

**1. Que représente l'« énergie » dans ce carnet, et quelle est sa valeur initiale ?**

<details><summary>Réponse</summary>

L'énergie est le **nombre de conflits** de la grille (cases en désaccord avec une ligne, une colonne ou un bloc). La grille complétée par permutations démarre à une **énergie positive** ; le recuit cherche à la faire descendre à **0** (une solution valide). C'est cette fonction de coût simple que le recuit explore.

</details>

**2. Que montre le premier échange manuel sur la valeur de l'énergie ?**

<details><summary>Réponse</summary>

Le premier échange démontré (deux colonnes dans une ligne) fait **augmenter** l'énergie de 1 : l'échange est *défavorable*. Le point est que le recuit **accepte parfois** un tel écart (selon la température) — c'est précisément ce qui lui permet de franchir les plateaux et d'échapper aux minima locaux qu'une descente de gradient pure subirait.

</details>

**3. Pourquoi le recuit est-il si lent face au backtracking sur le puzzle difficile ?**

<details><summary>Réponse</summary>

Le backtracking **cible** la solution : sur le puzzle 3 (51 cases vides) il la trouve en **295 appels / 0,0 ms affichés**. Le recuit, lui, *cherche* en minimisant les conflits sans savoir où est la solution — il lui faut **296 ms** sur le même puzzle. C'est le prix d'une méthode heuristique généraliste : robuste, mais sans la coupure exacte (pruning) qui fait la force du backtracking sur un problème à contraintes.

</details>

## Pour aller plus loin

- **[Sudoku-05 — PSO](Sudoku-05-PSO-Python.ipynb)** : le carnet suivant — l'optimisation par essaims de particules, même philosophie « minimiser les conflits ».
- **Jumeau Python** → [Sudoku-04-SimulatedAnnealing-Python](Sudoku-04-SimulatedAnnealing-Python.ipynb) — le carnet d'origine, en Python.

- [Sudoku-04 — Recuit Simulé (C#)](Sudoku-04-SimulatedAnnealing-CSharp.ipynb) : le jumeau C# du même recuit.

### Lectures

- Kirkpatrick, Gelatt & Vecchi (1983), *Optimization by Simulated Annealing*, Science — l'algorithme originel.

> Carnet issu de l'EPIC #26534.
